# Für Tris
### Hier findest du 2 Codes, einmal diesen, der auf deinem Code basiert und bei mir wegen dieses Speicherplatzproblems nicht funktioniert hat. 

# Einmal diese hier

In [2]:
# Cell 1: Imports and Configuration
# =================================
import ast
import threading
from flask import Flask, request, render_template_string
import pandas as pd
import numpy as np
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

# Configuration
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "argentic"

# CSV file paths
CITIES_CSV = "adjusted_datasets/adjusted_cities.csv"
FLIGHTS_CSV = "adjusted_datasets/adjusted_flights.csv"
HOTELS_CSV = "adjusted_datasets/adjusted_hotels.csv"
RESTAURANTS_CSV = "adjusted_datasets/adjusted_restaurants.csv"
PREFERENCES_CSV = "adjusted_datasets/preferences.csv"
USERS_CSV = "adjusted_datasets/users.csv"
PASSPORTS_CSV = "adjusted_datasets/adjusted_passports.csv"
HISTORIES_CSV = "adjusted_datasets/histories.csv"

# Initialize Flask app
app = Flask(__name__)
conversation_history = []

# Connect to Neo4j
try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    with driver.session() as session:
        session.run("RETURN 1")
    print("Successfully connected to Neo4j.")
except ServiceUnavailable as e:
    print("Neo4j connection error:", e)
    exit(1)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit


Neo4j connection error: Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [WinError 10061] Es konnte keine Verbindung hergestellt werden, da der Zielcomputer die Verbindung verweigerte)
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [WinError 10061] Es konnte keine Verbindung hergestellt werden, da der Zielcomputer die Verbindung verweigerte)


In [ ]:
# Cell 2: Graph Construction
# ==========================
def create_node(tx, label, id_field, props):
    query = f"MERGE (n:{label} {{{id_field}: ${id_field}}}) SET n += $props"
    tx.run(query, **{id_field: props[id_field]}, props=props)

def create_relationship(tx, label_from, key_from, value_from, rel_type, label_to, key_to, value_to):
    query = f"""
    MATCH (a:{label_from} {{{key_from}: $value_from}})
    MATCH (b:{label_to} {{{key_to}: $value_to}})
    MERGE (a)-[r:{rel_type}]->(b)
    """
    tx.run(query, value_from=value_from, value_to=value_to)

def build_graph():
    with driver.session() as session:
        # Load all CSV files
        dfs = {
            'City': pd.read_csv(CITIES_CSV),
            'Flight': pd.read_csv(FLIGHTS_CSV),
            'Hotel': pd.read_csv(HOTELS_CSV),
            'Restaurant': pd.read_csv(RESTAURANTS_CSV),
            'Preference': pd.read_csv(PREFERENCES_CSV),
            'User': pd.read_csv(USERS_CSV),
            'Passport': pd.read_csv(PASSPORTS_CSV),
            'History': pd.read_csv(HISTORIES_CSV)
        }
        
        # Create all nodes
        for label, df in dfs.items():
            id_field = f"{label.lower()}_id"
            for _, row in df.iterrows():
                session.execute_write(create_node, label, id_field, row.to_dict())
        
        # Create relationships
        for _, row in dfs['History'].iterrows():
            hist_id = row.get("history_id")
            
            if pd.notna(row.get("hotels")):
                try:
                    hotel_ids = ast.literal_eval(row["hotels"])
                    for h_id in hotel_ids:
                        session.execute_write(
                            create_relationship,
                            "History", "history_id", hist_id,
                            "STAYED_AT", "Hotel", "hotel_id", h_id
                        )
                except Exception as e:
                    print(f"Error parsing hotels for history {hist_id}: {e}")
            
            if pd.notna(row.get("restaurants")):
                try:
                    rest_ids = ast.literal_eval(row["restaurants"])
                    for r_id in rest_ids:
                        session.execute_write(
                            create_relationship,
                            "History", "history_id", hist_id,
                            "DINED_AT", "Restaurant", "restaurant_id", r_id
                        )
                except Exception as e:
                    print(f"Error parsing restaurants for history {hist_id}: {e}")
        
        # Create preference relationships
        for _, row in dfs['Preference'].iterrows():
            pref_id = row.get("preference_id")
            
            if pd.notna(row.get("top_cities")):
                try:
                    cities = ast.literal_eval(row["top_cities"])
                    for city in cities:
                        session.execute_write(
                            create_relationship,
                            "Preference", "preference_id", pref_id,
                            "HAS_CITY_PREFERENCE", "City", "City", city
                        )
                except Exception as e:
                    print(f"Error parsing top_cities for preference {pref_id}: {e}")
            
            if pd.notna(row.get("top_hotels")):
                try:
                    hotels = ast.literal_eval(row["top_hotels"])
                    for h_id in hotels:
                        session.execute_write(
                            create_relationship,
                            "Preference", "preference_id", pref_id,
                            "HAS_HOTEL_PREFERENCE", "Hotel", "hotel_id", h_id
                        )
                except Exception as e:
                    print(f"Error parsing top_hotels for preference {pref_id}: {e}")
            
            if pd.notna(row.get("top_restaurants")):
                try:
                    restaurants = ast.literal_eval(row["top_restaurants"])
                    for r_id in restaurants:
                        session.execute_write(
                            create_relationship,
                            "Preference", "preference_id", pref_id,
                            "HAS_RESTAURANT_PREFERENCE", "Restaurant", "restaurant_id", r_id
                        )
                except Exception as e:
                    print(f"Error parsing top_restaurants for preference {pref_id}: {e}")
            
            if pd.notna(row.get("visa_preference")):
                visa_pref = row["visa_preference"]
                for _, pport_row in dfs['Passport'].iterrows():
                    if pd.notna(pport_row.get("Requirement")) and visa_pref.strip() == pport_row["Requirement"].strip():
                        session.execute_write(
                            create_relationship,
                            "Preference", "preference_id", pref_id,
                            "IS_READY_TO_APPLY_VISA", "Passport", "passport_id", pport_row["passport_id"]
                        )
        
        # Create visa relationships
        for _, hist_row in dfs['History'].iterrows():
            if pd.notna(hist_row.get("issued_passport")):
                issued_p = hist_row["issued_passport"]
                for _, pport_row in dfs['Passpo'
                'rt'].iterrows():
                    if pd.notna(pport_row.get("Origin")) and issued_p.strip() == pport_row["Origin"].strip():
                        session.execute_write(
                            create_relationship,
                            "Passport", "passport_id", pport_row["passport_id"],
                            "REQUIRED_VISA_LIKE", "History", "history_id", hist_row["history_id"]
                        )
    
    print("Graph successfully built with all nodes and relationships!")

# Build the graph (run once)
build_graph()

Graph successfully built with all nodes and relationships!


In [3]:
# Cell 3: Knowledge Base Construction
# ==================================
def build_representation(props, fields):
    parts = []
    for field, label in fields.items():
        value = props.get(field)
        if value is not None and str(value).strip() != "":
            parts.append(f"{label}: {value}")
    return "; ".join(parts)

representation_functions = {
    "City": lambda n: build_representation(n._properties, {
        "city_id": "ID", "City": "City", "Country": "Country",
        "Climate: Average number of sunshine hours": "Sunshine hours",
        "Food: Average cost of a meal": "Avg meal cost",
        "Tourist attractions: Number of 'Things to do' on Tripadvisor": "Attractions"
    }),
    "Flight": lambda n: build_representation(n._properties, {
        "flight_id": "ID", "Airline": "Airline", 
        "Total Fare (EUR)": "Price (EUR)", "Duration (hrs)": "Duration",
        "Departure Airport Code": "From", "Arrival Airport Code": "To"
    }),
    "Hotel": lambda n: build_representation(n._properties, {
        "hotel_id": "ID", "name": "Name", "price": "Price",
        "City": "City", "Country": "Country", "number_reviews": "Reviews"
    }),
    "Restaurant": lambda n: build_representation(n._properties, {
        "restaurant_id": "ID", "Restaurant Name": "Name", 
        "Cuisines": "Cuisines", "Average Cost for two": "Avg cost for two",
        "City": "City", "Country": "Country", "Aggregate rating": "Rating"
    }),
    "Preference": lambda n: build_representation(n._properties, {
        "preference_id": "ID", "visa_preference": "Visa pref",
        "preferred_flight_price_range": "Flight budget",
        "preferred_hotel_price_range": "Hotel budget",
        "preferred_cuisines": "Favorite cuisines"
    }),
    "User": lambda n: build_representation(n._properties, {
        "User_ID": "ID", "Username": "Username", 
        "City": "From", "Country": "Country"
    }),
    "Passport": lambda n: build_representation(n._properties, {
        "passport_id": "ID", "Origin": "Nationality",
        "Requirement": "Visa requirement"
    }),
    "History": lambda n: build_representation(n._properties, {
        "history_id": "ID", "city": "Visited city", 
        "country": "Country", "length_of_stay": "Days stayed"
    })
}

# Build the knowledge base
with driver.session() as session:
    all_representations = []
    for label, func in representation_functions.items():
        result = session.run(f"MATCH (n:{label}) RETURN n")
        for record in result:
            rep = func(record["n"])
            if rep:
                all_representations.append(rep)
    
    representations = list(set(all_representations))
    print(f"Created {len(representations)} unique representations for retrieval")

# Compute embeddings
print("Computing embeddings...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embedder.encode(representations, convert_to_tensor=True)

Created 13836 unique representations for retrieval
Computing embeddings...


In [ ]:
# Cell 4: Enhanced Retrieval System
# ================================
def expand_query_with_context(query):
    query_embedding = embedder.encode([query], convert_to_tensor=True)
    cos_scores = cosine_similarity(query_embedding.cpu().numpy(), doc_embeddings.cpu().numpy())[0]
    top_indices = np.argsort(cos_scores)[-3:][::-1]
    
    expanded_terms = []
    for idx in top_indices:
        doc = representations[idx]
        terms = [word for word in doc.split() if word.isalpha() and len(word) > 3]
        expanded_terms.extend(terms[:3])
    
    travel_terms = []
    if "hotel" not in query.lower() and "hotel" in " ".join(expanded_terms).lower():
        travel_terms.append("hotel options")
    if "flight" not in query.lower() and "flight" in " ".join(expanded_terms).lower():
        travel_terms.append("flight options")
    if "restaurant" not in query.lower() and "restaurant" in " ".join(expanded_terms).lower():
        travel_terms.append("dining options")
    
    expanded_query = f"{query} {' '.join(set(expanded_terms))} {' '.join(travel_terms)}"
    return expanded_query.strip()

def retrieve_relevant_documents(query, top_k=10, similarity_threshold=0.3):
    expanded_query = expand_query_with_context(query)
    query_embedding = embedder.encode([expanded_query], convert_to_tensor=True)
    cos_scores = cosine_similarity(query_embedding.cpu().numpy(), doc_embeddings.cpu().numpy())[0]
    
    sorted_indices = np.argsort(cos_scores)[::-1]
    filtered_indices = [i for i in sorted_indices if cos_scores[i] > similarity_threshold]
    
    if len(filtered_indices) < top_k:
        filtered_indices = sorted_indices[:top_k]
    
    retrieved_docs = [representations[i] for i in filtered_indices]
    return retrieved_docs, expanded_query

: 

In [ ]:
# Cell 5: Intelligent Response Generation (Multilingual Support)
# =============================================================
# Initialize the text generation pipeline
try:
    generator = pipeline(
        "text-generation",
        model="distilgpt2",
        do_sample=True,
        temperature=0.7,
        max_new_tokens=120,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2
    )
    print("Model loaded successfully")
except Exception as e:
    print(f"Couldn't load model: {e}")
    generator = None

def detect_query_type(query):
    query = query.lower()
    if any(word in query for word in ["complete", "full", "komplett", "ganze", "reise", "trip"]):
        return "complete_trip"
    elif any(word in query for word in ["hotel", "unterkunft", "übernachtung"]):
        return "hotel"
    elif any(word in query for word in ["restaurant", "essen", "dining", "küche"]):
        return "restaurant"
    elif any(word in query for word in ["flight", "flug", "airline"]):
        return "flight"
    else:
        return "general"

def detect_query_language(query):
    german_words = ["der", "die", "das", "und", "ich", "du", "wir", "essen", "hotel", "flug"]
    if any(word in query.lower() for word in german_words):
        return "german"
    return "english"

def format_section(title, items, lang="english"):
    if not items:
        return ""
    
    if lang == "german":
        header = f"=== {title} ==="
    else:
        header = f"=== {title} ==="
    
    return header + "\n" + "\n".join(f"- {item}" for item in items[:5]) + "\n\n"

def generate_travel_recommendation(query):
    travel_keywords = [
        "travel", "flight", "hotel", "restaurant", "visa", "passport", 
        "trip", "destination", "city", "country", "vacation", "itinerary",
        "reise", "flug", "hotel", "restaurant", "visum", "pass", 
        "urlaub", "ziel", "stadt", "land", "reiseplan"
    ]
    
    lang = detect_query_language(query)
    
    if not any(kw in query.lower() for kw in travel_keywords):
        if lang == "german":
            return "Ich spezialisiere mich auf Reiseempfehlungen. Könnten Sie bitte eine Reisefrage stellen?", []
        else:
            return "I specialize in travel recommendations. Could you please ask a travel-related question?", []
    
    query_type = detect_query_type(query)
    retrieved_docs, expanded_query = retrieve_relevant_documents(query)
    
    if not retrieved_docs:
        if lang == "german":
            return "Ich konnte keine relevanten Reiseinformationen finden. Könnten Sie es anders formulieren?", []
        else:
            return "I couldn't find relevant travel information. Could you try rephrasing?", []
    
    if lang == "german":
        response = "Hier sind meine Reiseempfehlungen für Sie:\n\n"
    else:
        response = "Here are my travel recommendations for you:\n\n"
    
    # Filter and organize recommendations by type
    cities = [d for d in retrieved_docs if "City:" in d]
    flights = [d for d in retrieved_docs if "Flight:" in d]
    hotels = [d for d in retrieved_docs if "Hotel:" in d]
    restaurants = [d for d in retrieved_docs if "Restaurant:" in d]
    
    if query_type == "complete_trip":
        response += format_section("Recommended Destinations", cities, lang)
        response += format_section("Flight Options", flights, lang)
        response += format_section("Hotel Suggestions", hotels, lang)
        response += format_section("Dining Recommendations", restaurants, lang)
    elif query_type == "hotel":
        response += format_section("Hotel Recommendations", hotels, lang)
    elif query_type == "restaurant":
        response += format_section("Restaurant Recommendations", restaurants, lang)
    elif query_type == "flight":
        response += format_section("Flight Options", flights, lang)
    else:
        response += "\n".join(retrieved_docs[:8])
    
    # Add budget estimation if it's a complete trip query
    if query_type == "complete_trip":
        if lang == "german":
            response += "\nGeschätztes Budget für diese Reise: 800-1200€"
        else:
            response += "\nEstimated budget for this trip: 800-1200€"
    
    return response, retrieved_docs

In [ ]:
# Cell 6: Flask Web Interface
# ==========================
HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>Advanced Travel Assistant</title>
    <style>
        body { 
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            max-width: 900px; 
            margin: 0 auto; 
            padding: 20px;
            background-color: #f5f7fa;
            color: #333;
        }
        h1 { 
            color: #2c3e50; 
            text-align: center; 
            margin-bottom: 20px;
        }
        .filter-container {
            display: flex;
            gap: 10px;
            margin-bottom: 20px;
            flex-wrap: wrap;
        }
        .filter-btn {
            padding: 8px 15px;
            background-color: #3498db;
            color: white;
            border: none;
            border-radius: 20px;
            cursor: pointer;
            transition: all 0.3s;
        }
        .filter-btn:hover {
            background-color: #2980b9;
            transform: translateY(-2px);
        }
        .filter-btn.active {
            background-color: #2c3e50;
        }
        #chat-container { 
            border: 1px solid #ddd; 
            border-radius: 10px; 
            padding: 15px; 
            height: 400px; 
            overflow-y: auto;
            margin-bottom: 20px; 
            background: white;
            box-shadow: 0 2px 10px rgba(0,0,0,0.05);
        }
        .message { 
            margin: 10px 0; 
            padding: 12px 15px; 
            border-radius: 8px;
            max-width: 80%;
            word-wrap: break-word;
        }
        .user { 
            background: #e3f2fd; 
            margin-left: auto;
            border-bottom-right-radius: 0;
        }
        .bot { 
            background: #f1f1f1; 
            margin-right: auto;
            border-bottom-left-radius: 0;
        }
        .system-message { 
            font-size: 12px; 
            color: #7f8c8d; 
            text-align: center; 
            margin: 10px 0;
        }
        #retrieved-data { 
            border: 1px dashed #95a5a6; 
            border-radius: 8px;
            padding: 15px; 
            margin-bottom: 20px; 
            background: #fffde7; 
            font-size: 14px;
            display: none;
        }
        #query-form { 
            display: flex; 
            margin-bottom: 20px; 
            gap: 10px;
        }
        #question-input { 
            flex-grow: 1; 
            padding: 12px; 
            border: 1px solid #ddd; 
            border-radius: 8px;
            font-size: 16px;
            box-shadow: inset 0 1px 3px rgba(0,0,0,0.1);
        }
        #submit-btn { 
            padding: 12px 25px;
            background: #2c3e50; 
            color: white;
            border: none; 
            border-radius: 8px; 
            cursor: pointer;
            font-weight: bold;
            transition: background 0.3s;
        }
        #submit-btn:hover {
            background: #1a252f;
        }
        .toggle-data-btn {
            background: #95a5a6;
            color: white;
            border: none;
            padding: 8px 15px;
            border-radius: 5px;
            cursor: pointer;
            margin-bottom: 10px;
        }
        .query-examples {
            background: #e8f4f8;
            padding: 15px;
            border-radius: 8px;
            margin-bottom: 20px;
        }
        .example-query {
            color: #2980b9;
            cursor: pointer;
            margin: 5px 0;
            padding: 5px;
            border-radius: 4px;
        }
        .example-query:hover {
            background: #d6eaf8;
        }
    </style>
</head>
<body>
    <h1>Advanced Travel Assistant</h1>
    
    <div class="query-examples">
        <h3>Example Queries:</h3>
        <div class="example-query" onclick="fillQuery('I want a 7-day summer trip focusing on food')">I want a 7-day summer trip focusing on food</div>
        <div class="example-query" onclick="fillQuery('Recommend good hotels in Istanbul')">Recommend good hotels in Istanbul</div>
        <div class="example-query" onclick="fillQuery('Which restaurants in New York are affordable?')">Which restaurants in New York are affordable?</div>
        <div class="example-query" onclick="fillQuery('Show me flights to Europe under €500')">Show me flights to Europe under €500</div>
        <div class="example-query" onclick="fillQuery('Ich möchte eine 7-tägige Sommerreise mit Fokus auf Essen')">Ich möchte eine 7-tägige Sommerreise mit Fokus auf Essen</div>
        <div class="example-query" onclick="fillQuery('Empfehle mir gute Hotels in Istanbul')">Empfehle mir gute Hotels in Istanbul</div>
    </div>
    
    <div class="filter-container">
        <button class="filter-btn active" onclick="setQueryType('all')">Complete Trip</button>
        <button class="filter-btn" onclick="setQueryType('hotel')">Hotels Only</button>
        <button class="filter-btn" onclick="setQueryType('restaurant')">Restaurants Only</button>
        <button class="filter-btn" onclick="setQueryType('flight')">Flights Only</button>
    </div>
    
    <div id="chat-container">
        <div class="system-message">Welcome! Ask me about travel destinations, flights, hotels, restaurants, or visa requirements.</div>
    </div>
    
    <button class="toggle-data-btn" onclick="toggleData()">Show/Hide Data Sources</button>
    <div id="retrieved-data">
        <h3>Used Data Sources:</h3>
        <div id="data-sources"></div>
    </div>
    
    <form method="post" id="query-form" onsubmit="return handleSubmit()">
        <input type="hidden" id="query-type" name="query_type" value="all">
        <input type="text" name="question" id="question-input" 
               placeholder="E.g., 'I want an affordable city trip in spring'" required>
        <input type="submit" id="submit-btn" value="Ask">
    </form>
    
    <script>
        var chatContainer = document.getElementById("chat-container");
        chatContainer.scrollTop = chatContainer.scrollHeight;
        
        document.getElementById("question-input").focus();
        
        function setQueryType(type) {
            document.getElementById("query-type").value = type;
            const buttons = document.querySelectorAll(".filter-btn");
            buttons.forEach(btn => btn.classList.remove("active"));
            event.target.classList.add("active");
        }
        
        function fillQuery(query) {
            document.getElementById("question-input").value = query;
            document.getElementById("question-input").focus();
        }
        
        function toggleData() {
            const dataDiv = document.getElementById("retrieved-data");
            dataDiv.style.display = dataDiv.style.display === "none" ? "block" : "none";
        }
        
        function handleSubmit() {
            const query = document.getElementById("question-input").value;
            const queryType = document.getElementById("query-type").value;
            
            const chatDiv = document.getElementById("chat-container");
            const userMsg = document.createElement("div");
            userMsg.className = "message user";
            userMsg.innerHTML = "<strong>User:</strong> " + query;
            chatDiv.appendChild(userMsg);
            
            chatContainer.scrollTop = chatContainer.scrollHeight;
            
            fetch("/", {
                method: "POST",
                headers: {
                    "Content-Type": "application/x-www-form-urlencoded",
                },
                body: `question=${encodeURIComponent(query)}&query_type=${queryType}`
            })
            .then(response => response.text())
            .then(html => {
                const parser = new DOMParser();
                const doc = parser.parseFromString(html, "text/html");
                
                const history = doc.querySelectorAll("#chat-container .message");
                const lastMsg = history[history.length - 1];
                chatDiv.appendChild(lastMsg.cloneNode(true));
                
                const dataSources = doc.getElementById("retrieved-data");
                if (dataSources) {
                    document.getElementById("data-sources").innerHTML = dataSources.innerHTML;
                }
                
                chatContainer.scrollTop = chatContainer.scrollHeight;
            });
            
            document.getElementById("question-input").value = "";
            return false;
        }
    </script>
</body>
</html>
"""

In [ ]:
# Cell 7: Flask Routes and App Launch
# ==================================
@app.route("/", methods=["GET", "POST"])
def index():
    global conversation_history
    
    if not conversation_history:
        conversation_history.append({
            "sender": "system",
            "text": "Welcome! Ask me about travel destinations, flights, hotels, restaurants, or visa requirements."
        })
    
    retrieved_data = []
    if request.method == "POST":
        question = request.form["question"]
        query_type = request.form.get("query_type", "all")
        
        if query_type == "hotel":
            question = f"Hotels only: {question}"
        elif query_type == "restaurant":
            question = f"Restaurants only: {question}"
        elif query_type == "flight":
            question = f"Flights only: {question}"
        
        conversation_history.append({"sender": "User", "text": question})
        answer, retrieved_data = generate_travel_recommendation(question)
        conversation_history.append({"sender": "Bot", "text": answer})
    
    return render_template_string(HTML_TEMPLATE, 
                                history=conversation_history,
                                retrieved_data=retrieved_data)

def run_flask_app():
    app.run(port=5001, debug=False, use_reloader=False)

threading.Thread(target=run_flask_app, daemon=True).start()
print("Travel Assistant is running at http://127.0.0.1:5001")

Travel Assistant is running at http://127.0.0.1:5001


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit


# Zweite code der bisschen lange text züruck gibst 

In [ ]:
# Cell 1: Setup and Imports
import ast
import threading
from flask import Flask, request, render_template_string
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

app = Flask(__name__)

# CSV file paths (all in the adjusted_datasets folder)
CITIES_CSV = "adjusted_datasets/adjusted_cities.csv"
FLIGHTS_CSV = "adjusted_datasets/adjusted_flights.csv"
HOTELS_CSV = "adjusted_datasets/adjusted_hotels.csv"
RESTAURANTS_CSV = "adjusted_datasets/adjusted_restaurants.csv"
PREFERENCES_CSV = "adjusted_datasets/preferences.csv"
USERS_CSV = "adjusted_datasets/users.csv"
PASSPORTS_CSV = "adjusted_datasets/adjusted_passports.csv"
HISTORIES_CSV = "adjusted_datasets/histories.csv"

# Global conversation history for the chat interface
conversation_history = []

# ----------------------------
# 2. Data Loading (without Neo4j)
# ----------------------------
def load_data():
    # Load all CSV files into DataFrames
    cities_df = pd.read_csv(CITIES_CSV)
    flights_df = pd.read_csv(FLIGHTS_CSV)
    hotels_df = pd.read_csv(HOTELS_CSV)
    restaurants_df = pd.read_csv(RESTAURANTS_CSV)
    preferences_df = pd.read_csv(PREFERENCES_CSV)
    users_df = pd.read_csv(USERS_CSV)
    passports_df = pd.read_csv(PASSPORTS_CSV)
    histories_df = pd.read_csv(HISTORIES_CSV)
    
    # Convert DataFrames to dictionaries for easier access
    data = {
        "cities": cities_df.to_dict('records'),
        "flights": flights_df.to_dict('records'),
        "hotels": hotels_df.to_dict('records'),
        "restaurants": restaurants_df.to_dict('records'),
        "preferences": preferences_df.to_dict('records'),
        "users": users_df.to_dict('records'),
        "passports": passports_df.to_dict('records'),
        "histories": histories_df.to_dict('records')
    }
    return data

# Load all data
data = load_data()
print("Data loaded successfully!")

# ----------------------------
# 3. Representation and Retrieval
# ----------------------------
def build_representation(item, fields):
    parts = []
    for field, label in fields.items():
        value = item.get(field)
        if value is not None and str(value).strip() != "":
            parts.append(f"{label}: {value}")
    return "; ".join(parts)

def represent_city(city):
    fields = {
        "City": "City", "Country": "Country",
        "Remote connection: Average WiFi speed (Mbps per second)": "WiFi Speed",
        "Co-working spaces: Number of co-working spaces": "Co-working Spaces",
        "Accommodation: Average price of 1 bedroom apartment per month": "Apartment Price",
        "Food: Average cost of a meal at a local, mid-level restaurant": "Meal Cost",
        "Tourist attractions: Number of 'Things to do' on Tripadvisor": "Attractions"
    }
    return build_representation(city, fields)

def represent_flight(flight):
    fields = {
        "Airline": "Airline", "Total Fare (EUR)": "Price",
        "Departure Airport Code": "From", "Arrival Airport Code": "To",
        "Duration (hrs)": "Duration", "Class": "Class"
    }
    return build_representation(flight, fields)

def represent_hotel(hotel):
    fields = {
        "name": "Name", "price": "Price",
        "number_reviews": "Reviews", "City": "City"
    }
    return build_representation(hotel, fields)

def represent_restaurant(restaurant):
    fields = {
        "Restaurant Name": "Name", "Cuisines": "Cuisines",
        "Average Cost for two": "Price for Two", "City": "City"
    }
    return build_representation(restaurant, fields)

# Create representations for all data
representations = []
for city in data["cities"]:
    representations.append(represent_city(city))
for flight in data["flights"]:
    representations.append(represent_flight(flight))
for hotel in data["hotels"]:
    representations.append(represent_hotel(hotel))
for restaurant in data["restaurants"]:
    representations.append(represent_restaurant(restaurant))

representations = list(set(representations))
print("Total representations for retrieval:", len(representations))

# ----------------------------
# 4. Embeddings and Retrieval
# ----------------------------
print("Computing embeddings...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embedder.encode(representations, convert_to_tensor=True)

def retrieve_documents(query, top_k=8, similarity_threshold=0.0):
    query_embedding = embedder.encode([query], convert_to_tensor=True)
    cos_scores = cosine_similarity(query_embedding.cpu().numpy(), doc_embeddings.cpu().numpy())[0]
    sorted_indices = np.argsort(cos_scores)[::-1]
    retrieved_docs = [representations[i] for i in sorted_indices[:top_k]]
    return retrieved_docs

# ----------------------------
# 5. Query Processing and Generation
# ----------------------------
generator = pipeline(
    "text-generation",
    model="gpt2",
    do_sample=True,
    temperature=0.7,
    max_new_tokens=100,
    no_repeat_ngram_size=3,
    repetition_penalty=1.2
)

def detect_query_type(query):
    query_lower = query.lower()
    if any(word in query_lower for word in ["hotel", "stay", "accommodation", "lodging"]):
        return "hotel"
    elif any(word in query_lower for word in ["restaurant", "eat", "dine", "food", "cuisine"]):
        return "restaurant"
    elif any(word in query_lower for word in ["flight", "fly", "airline", "ticket"]):
        return "flight"
    elif any(word in query_lower for word in ["city", "destination", "place", "visit", "location"]):
        return "city"
    elif any(word in query_lower for word in ["trip", "itinerary", "plan", "vacation", "holiday"]):
        return "complete_trip"
    else:
        return "general"

def refine_query(raw_query):
    query_type = detect_query_type(raw_query)
    prompt = f"""
    Refine this travel query to be more specific for a {query_type} search:
    Original Query: {raw_query}
    Refined Query:"""
    result = generator(prompt, num_return_sequences=1)
    refined = result[0]["generated_text"].replace(prompt, "").strip().split("\n")[0].strip()
    return refined, query_type

# ----------------------------
# 6. Response Generation
# ----------------------------
def generate_response(query):
    # First detect what kind of information the user wants
    refined_query, query_type = refine_query(query)
    print(f"Detected query type: {query_type}, Refined: {refined_query}")
    
    # Retrieve relevant documents based on query type
    retrieved_docs = retrieve_documents(refined_query, top_k=10)
    
    if not retrieved_docs:
        return "I couldn't find enough information about that. Could you be more specific?", []
    
    # Generate a prompt based on query type
    if query_type == "hotel":
        prompt = f"""Based on these hotel options:
        {retrieved_docs}
        
        Provide a summary of the best hotel options for someone asking: {query}
        Consider price, reviews, and location. Be concise but helpful."""
        
    elif query_type == "restaurant":
        prompt = f"""Based on these restaurant options:
        {retrieved_docs}
        
        Recommend good dining options for someone asking: {query}
        Consider cuisine types, prices, and locations."""
        
    elif query_type == "flight":
        prompt = f"""Based on these flight options:
        {retrieved_docs}
        
        Summarize the best flight options for someone asking: {query}
        Consider price, duration, and airline quality."""
        
    elif query_type == "city":
        prompt = f"""Based on these city details:
        {retrieved_docs}
        
        Describe the best travel destinations for someone asking: {query}
        Highlight attractions, costs, and unique features."""
        
    elif query_type == "complete_trip":
        prompt = f"""Based on this travel data:
        {retrieved_docs}
        
        Create a complete trip itinerary for someone asking: {query}
        Include recommendations for flights, hotels, restaurants, and activities.
        Structure it as a day-by-day plan with estimated costs."""
        
    else:
        prompt = f"""Based on this travel data:
        {retrieved_docs}
        
        Provide a comprehensive travel answer to: {query}
        Include relevant details about destinations, accommodations, and activities."""
    
    # Generate the response
    result = generator(prompt, num_return_sequences=1)
    response = result[0]["generated_text"].replace(prompt, "").strip()
    
    return response, retrieved_docs

# ----------------------------
# 7. Enhanced Flask Web Interface
# ----------------------------
HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>Enhanced Travel Assistant</title>
    <style>
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            max-width: 1200px;
            margin: 0 auto;
            padding: 20px;
            background-color: #f5f7fa;
            color: #333;
        }
        .header {
            background-color: #4285f4;
            color: white;
            padding: 20px;
            border-radius: 8px;
            margin-bottom: 20px;
            text-align: center;
        }
        .filter-section {
            background-color: white;
            padding: 15px;
            border-radius: 8px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        .filter-buttons {
            display: flex;
            gap: 10px;
            flex-wrap: wrap;
            margin-top: 10px;
        }
        .filter-button {
            padding: 8px 15px;
            background-color: #e0e0e0;
            border: none;
            border-radius: 20px;
            cursor: pointer;
            transition: background-color 0.3s;
        }
        .filter-button:hover {
            background-color: #d0d0d0;
        }
        .filter-button.active {
            background-color: #4285f4;
            color: white;
        }
        .chat-container {
            background-color: white;
            border-radius: 8px;
            padding: 20px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            height: 400px;
            overflow-y: auto;
        }
        .message {
            margin-bottom: 15px;
            padding: 10px 15px;
            border-radius: 18px;
            max-width: 70%;
            word-wrap: break-word;
        }
        .user-message {
            background-color: #e3f2fd;
            margin-left: auto;
            border-bottom-right-radius: 4px;
        }
        .bot-message {
            background-color: #f1f1f1;
            margin-right: auto;
            border-bottom-left-radius: 4px;
        }
        .data-section {
            background-color: white;
            border-radius: 8px;
            padding: 20px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        .data-item {
            padding: 10px;
            border-bottom: 1px solid #eee;
        }
        .input-section {
            display: flex;
            gap: 10px;
        }
        #user-input {
            flex-grow: 1;
            padding: 12px;
            border: 1px solid #ddd;
            border-radius: 8px;
            font-size: 16px;
        }
        #submit-button {
            padding: 12px 20px;
            background-color: #4285f4;
            color: white;
            border: none;
            border-radius: 8px;
            cursor: pointer;
            font-size: 16px;
        }
        #submit-button:hover {
            background-color: #3367d6;
        }
        .query-type-indicator {
            font-size: 14px;
            color: #666;
            margin-top: 5px;
            font-style: italic;
        }
    </style>
</head>
<body>
    <div class="header">
        <h1>Enhanced Travel Assistant</h1>
        <p>Get personalized travel recommendations for flights, hotels, restaurants and destinations</p>
    </div>
    
    <div class="filter-section">
        <h3>Not sure what to ask? Try these:</h3>
        <div class="filter-buttons">
            <button class="filter-button" onclick="setQuery('Best hotels in New York under $200')">Hotels</button>
            <button class="filter-button" onclick="setQuery('Italian restaurants in London')">Restaurants</button>
            <button class="filter-button" onclick="setQuery('Cheapest flights to Dubai next month')">Flights</button>
            <button class="filter-button" onclick="setQuery('Best digital nomad cities with good WiFi')">Destinations</button>
            <button class="filter-button" onclick="setQuery('Plan a complete 5-day trip to Istanbul')">Complete Trip</button>
        </div>
    </div>
    
    <div class="chat-container" id="chat-container">
        {% for msg in history %}
            <div class="message {% if msg.sender == 'User' %}user-message{% else %}bot-message{% endif %}">
                <strong>{{ msg.sender }}:</strong> {{ msg.text }}
                {% if msg.query_type %}
                <div class="query-type-indicator">Detected as: {{ msg.query_type }}</div>
                {% endif %}
            </div>
        {% endfor %}
    </div>
    
    <div class="data-section">
        <h3>Recommendation Details</h3>
        {% if retrieved_data %}
            {% for doc in retrieved_data %}
                <div class="data-item">{{ doc }}</div>
            {% endfor %}
        {% else %}
            <div class="data-item">No data retrieved yet. Ask about hotels, restaurants, flights or destinations.</div>
        {% endif %}
    </div>
    
    <form method="post" class="input-section">
        <input type="text" id="user-input" name="question" placeholder="Ask about hotels, flights, restaurants or destinations..." required>
        <input type="submit" id="submit-button" value="Send">
    </form>
    
    <script>
        function setQuery(query) {
            document.getElementById('user-input').value = query;
            document.getElementById('user-input').focus();
        }
        
        // Auto-scroll chat to bottom
        var chatContainer = document.getElementById("chat-container");
        chatContainer.scrollTop = chatContainer.scrollHeight;
    </script>
</body>
</html>
"""

@app.route("/", methods=["GET", "POST"])
def index():
    global conversation_history
    retrieved_data = []
    
    if request.method == "POST":
        question = request.form["question"]
        query_type = detect_query_type(question)
        conversation_history.append({
            "sender": "User", 
            "text": question,
            "query_type": query_type.replace("_", " ").title()
        })
        
        answer, retrieved_data = generate_response(question)
        conversation_history.append({
            "sender": "Assistant", 
            "text": answer
        })
    
    return render_template_string(HTML_TEMPLATE, history=conversation_history, retrieved_data=retrieved_data)

# ----------------------------
# 8. Run the Application
# ----------------------------
def run_flask_app():
    app.run(port=5001, debug=True)

# Start the Flask app in a background thread
threading.Thread(target=run_flask_app, daemon=True).start()
print("Travel Assistant is running! Access it at: http://127.0.0.1:5001")

C:\Users\mihab\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
C:\Users\mihab\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data loaded successfully!
Total representations for retrieval: 12312
Computing embeddings...


Device set to use cpu


Travel Assistant is running! Access it at: http://127.0.0.1:5001


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
Exception in thread Thread-7 (run_flask_app):
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\mihab\AppData\Local\Temp\ipykernel_16456\2464752197.py", line 436, in run_flask_app
  File "C:\Users\mihab\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\flask\app.py", line 612, in run
    run_simple(t.cast(str, host), port, self, **options)
  File "C:\Users\mihab\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\werkzeug\serving.py",

# 3


In [1]:
# Cell 1: Setup and Imports
import ast
import threading
from flask import Flask, request, render_template_string
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

app = Flask(__name__)

# CSV file paths (all in the adjusted_datasets folder)
CITIES_CSV = "adjusted_datasets/adjusted_cities.csv"
FLIGHTS_CSV = "adjusted_datasets/adjusted_flights.csv"
HOTELS_CSV = "adjusted_datasets/adjusted_hotels.csv"
RESTAURANTS_CSV = "adjusted_datasets/adjusted_restaurants.csv"
PREFERENCES_CSV = "adjusted_datasets/preferences.csv"
USERS_CSV = "adjusted_datasets/users.csv"
PASSPORTS_CSV = "adjusted_datasets/adjusted_passports.csv"
HISTORIES_CSV = "adjusted_datasets/histories.csv"

# Global conversation history for the chat interface
conversation_history = []

# ----------------------------
# 2. Data Loading (without Neo4j)
# ----------------------------
def load_data():
    # Load all CSV files into DataFrames
    cities_df = pd.read_csv(CITIES_CSV)
    flights_df = pd.read_csv(FLIGHTS_CSV)
    hotels_df = pd.read_csv(HOTELS_CSV)
    restaurants_df = pd.read_csv(RESTAURANTS_CSV)
    preferences_df = pd.read_csv(PREFERENCES_CSV)
    users_df = pd.read_csv(USERS_CSV)
    passports_df = pd.read_csv(PASSPORTS_CSV)
    histories_df = pd.read_csv(HISTORIES_CSV)
    
    # Convert DataFrames to dictionaries for easier access
    data = {
        "cities": cities_df.to_dict('records'),
        "flights": flights_df.to_dict('records'),
        "hotels": hotels_df.to_dict('records'),
        "restaurants": restaurants_df.to_dict('records'),
        "preferences": preferences_df.to_dict('records'),
        "users": users_df.to_dict('records'),
        "passports": passports_df.to_dict('records'),
        "histories": histories_df.to_dict('records')
    }
    return data

# Load all data
data = load_data()
print("Data loaded successfully!")

# ----------------------------
# 3. Representation and Retrieval
# ----------------------------
def build_representation(item, fields):
    parts = []
    for field, label in fields.items():
        value = item.get(field)
        if value is not None and str(value).strip() != "":
            parts.append(f"{label}: {value}")
    return "; ".join(parts)

def represent_city(city):
    fields = {
        "City": "City", "Country": "Country",
        "Remote connection: Average WiFi speed (Mbps per second)": "WiFi Speed",
        "Co-working spaces: Number of co-working spaces": "Co-working Spaces",
        "Accommodation: Average price of 1 bedroom apartment per month": "Apartment Price",
        "Food: Average cost of a meal at a local, mid-level restaurant": "Meal Cost",
        "Tourist attractions: Number of Things to do on Tripadvisor": "Attractions"
    }
    return build_representation(city, fields)

def represent_flight(flight):
    fields = {
        "Airline": "Airline", "Total Fare (EUR)": "Price",
        "Departure Airport Code": "From", "Arrival Airport Code": "To",
        "Duration (hrs)": "Duration", "Class": "Class"
    }
    return build_representation(flight, fields)

def represent_hotel(hotel):
    fields = {
        "name": "Name", "price": "Price",
        "number_reviews": "Reviews", "City": "City",
        "rating": "Rating", "address": "Address"
    }
    return build_representation(hotel, fields)

def represent_restaurant(restaurant):
    fields = {
        "Restaurant Name": "Name", "Cuisines": "Cuisines",
        "Average Cost for two": "Price for Two", "City": "City",
        "Aggregate rating": "Rating", "Address": "Address"
    }
    return build_representation(restaurant, fields)

# Create representations for all data
representations = []
for city in data["cities"]:
    representations.append(represent_city(city))
for flight in data["flights"]:
    representations.append(represent_flight(flight))
for hotel in data["hotels"]:
    representations.append(represent_hotel(hotel))
for restaurant in data["restaurants"]:
    representations.append(represent_restaurant(restaurant))

representations = list(set(representations))
print("Total representations for retrieval:", len(representations))

# ----------------------------
# 4. Embeddings and Retrieval
# ----------------------------
print("Computing embeddings...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embedder.encode(representations, convert_to_tensor=True)

def retrieve_documents(query, top_k=8, similarity_threshold=0.0):
    query_embedding = embedder.encode([query], convert_to_tensor=True)
    cos_scores = cosine_similarity(query_embedding.cpu().numpy(), doc_embeddings.cpu().numpy())[0]
    sorted_indices = np.argsort(cos_scores)[::-1]
    retrieved_docs = [representations[i] for i in sorted_indices[:top_k]]
    return retrieved_docs

# ----------------------------
# 5. Query Processing and Generation
# ----------------------------
generator = pipeline(
    "text-generation",
    model="gpt2",
    do_sample=True,
    temperature=0.7,
    max_new_tokens=200,
    no_repeat_ngram_size=3,
    repetition_penalty=1.2
)

def detect_query_type(query):
    query_lower = query.lower()
    if any(word in query_lower for word in ["hotel", "stay", "accommodation", "lodging"]):
        return "hotel"
    elif any(word in query_lower for word in ["restaurant", "eat", "dine", "food", "cuisine"]):
        return "restaurant"
    elif any(word in query_lower for word in ["flight", "fly", "airline", "ticket"]):
        return "flight"
    elif any(word in query_lower for word in ["city", "destination", "place", "visit", "location"]):
        return "city"
    elif any(word in query_lower for word in ["trip", "itinerary", "plan", "vacation", "holiday"]):
        return "complete_trip"
    elif any(word in query_lower for word in ["clear", "reset", "delete", "erase"]):
        return "clear_history"
    else:
        return "general"

def refine_query(raw_query):
    query_type = detect_query_type(raw_query)
    if query_type == "clear_history":
        return raw_query, query_type
        
    prompt = f"""
    Refine this travel query to be more specific for a {query_type} search:
    Original Query: {raw_query}
    Refined Query:"""
    result = generator(prompt, num_return_sequences=1)
    refined = result[0]["generated_text"].replace(prompt, "").strip().split("\n")[0].strip()
    return refined, query_type

# ----------------------------
# 6. IMPROVED Response Generation
# ----------------------------
def generate_response(query):
    # First detect what kind of information the user wants
    refined_query, query_type = refine_query(query)
    print(f"Detected query type: {query_type}, Refined: {refined_query}")
    
    # Handle clear history command
    if query_type == "clear_history":
        global conversation_history
        conversation_history = []
        return "I've cleared our conversation history. How can I help you with your travel plans?", []
    
    # Retrieve relevant documents based on query type
    retrieved_docs = retrieve_documents(refined_query, top_k=10)
    
    if not retrieved_docs:
        return "I couldn't find enough information about that. Could you be more specific?", []
    
    # Generate a prompt based on query type
    if query_type == "hotel":
        # Find hotels matching the query (e.g., price range)
        target_city = None
        max_price = None
        if "new york" in query.lower():
            target_city = "New York"
        if "under" in query.lower() and "$" in query.lower():
            try:
                max_price = float(query.split("$")[1].split()[0])
            except:
                pass
        
        matching_hotels = []
        for hotel in data["hotels"]:
            if target_city and hotel.get("City", "").lower() != target_city.lower():
                continue
            try:
                hotel_price = float(hotel.get("price", 99999))
                if max_price and hotel_price > max_price:
                    continue
                matching_hotels.append(hotel)
            except:
                continue
        
        if not matching_hotels:
            # Find cheapest hotel if none match the price
            if target_city:
                city_hotels = [h for h in data["hotels"] if h.get("City", "").lower() == target_city.lower()]
                if city_hotels:
                    try:
                        cheapest = min(city_hotels, key=lambda x: float(x.get("price", 99999)))
                        response = f"I couldn't find hotels under ${max_price} in {target_city}. The cheapest option available is:\n\n🏨 {cheapest['name']}\n   - Price: ${cheapest['price']}\n   - Reviews: {cheapest.get('number_reviews', 'N/A')}\n   - Rating: {cheapest.get('rating', 'N/A')}\n   - Address: {cheapest.get('address', 'N/A')}\n\nWould you like more information about this or other options?"
                        return response, [represent_hotel(cheapest)]
                    except:
                        pass
            
            return f"I couldn't find any hotels matching your criteria in our database. Please try a different search.", []
        
        # Sort by price
        matching_hotels.sort(key=lambda x: float(x.get("price", 99999)))
        
        # Build response
        response = f"Here are the best hotel options in {target_city if target_city else 'our database'} under ${max_price if max_price else 'any price'}:\n\n"
        for i, hotel in enumerate(matching_hotels[:5]):  # Show top 5
            response += f"🏨 {hotel['name']}\n"
            response += f"   - Price: ${hotel['price']}\n"
            response += f"   - Reviews: {hotel.get('number_reviews', 'N/A')}\n"
            response += f"   - Rating: {hotel.get('rating', 'N/A')}\n"
            response += f"   - Address: {hotel.get('address', 'N/A')}\n\n"
        
        response += "Would you like:\n"
        response += "1. More details about any of these hotels\n"
        response += "2. Cheaper options in a different area\n"
        response += "3. Higher-end options with better amenities\n"
        response += "4. Something else?"
        
        return response, [represent_hotel(h) for h in matching_hotels[:5]]
        
    elif query_type == "restaurant":
        # Find restaurants matching the query
        target_city = None
        cuisine_type = None
        
        # Extract city if mentioned
        for city in data["cities"]:
            if city['City'].lower() in query.lower():
                target_city = city['City']
                break
                
        # Extract cuisine type if mentioned
        cuisine_words = ["italian", "chinese", "french", "japanese", "mexican", "indian", "thai"]
        for word in cuisine_words:
            if word in query.lower():
                cuisine_type = word
                break
        
        matching_restaurants = []
        for restaurant in data["restaurants"]:
            if target_city and restaurant.get("City", "").lower() != target_city.lower():
                continue
            if cuisine_type and cuisine_type not in restaurant.get("Cuisines", "").lower():
                continue
            matching_restaurants.append(restaurant)
        
        if not matching_restaurants:
            return f"I couldn't find any {cuisine_type + ' ' if cuisine_type else ''}restaurants matching your criteria in {target_city if target_city else 'our database'}. Please try a different search.", []
        
        # Sort by price
        matching_restaurants.sort(key=lambda x: float(x.get("Average Cost for two", 0)))
        
        response = f"Here are some excellent {cuisine_type if cuisine_type else ''} restaurant options in {target_city if target_city else 'various cities'}:\n\n"
        for i, restaurant in enumerate(matching_restaurants[:5]):
            response += f"🍽️ {restaurant['Restaurant Name']}\n"
            response += f"   - Cuisine: {restaurant.get('Cuisines', 'N/A')}\n"
            response += f"   - Avg. cost for two: ${restaurant.get('Average Cost for two', 'N/A')}\n"
            response += f"   - Rating: {restaurant.get('Aggregate rating', 'N/A')}\n"
            response += f"   - Address: {restaurant.get('Address', 'N/A')}\n\n"
        
        response += "Would you like to:\n"
        response += "1. Filter by a specific price range\n"
        response += "2. See options in a different area\n"
        response += "3. Get recommendations for a different cuisine\n"
        response += "4. More details about any of these"
        
        return response, [represent_restaurant(r) for r in matching_restaurants[:5]]
        
    elif query_type == "flight":
        # Find flights matching the query
        target_destination = None
        max_price = None
        
        # Extract destination if mentioned
        for city in data["cities"]:
            if city['City'].lower() in query.lower():
                target_destination = city['City']
                break
                
        # Extract max price if mentioned
        if "under" in query.lower() and "$" in query.lower():
            try:
                max_price = float(query.split("$")[1].split()[0])
            except:
                pass
        
        matching_flights = []
        for flight in data["flights"]:
            if target_destination and flight.get("Arrival Airport Code", "").lower() != target_destination.lower():
                continue
            try:
                flight_price = float(flight.get("Total Fare (EUR)", 99999))
                if max_price and flight_price > max_price:
                    continue
                matching_flights.append(flight)
            except:
                continue
        
        if not matching_flights:
            return f"I couldn't find any flights matching your criteria. Please try a different search.", []
        
        # Sort by price
        matching_flights.sort(key=lambda x: float(x.get("Total Fare (EUR)", 99999)))
        
        response = f"Here are the best flight options to {target_destination if target_destination else 'various destinations'}:\n\n"
        for i, flight in enumerate(matching_flights[:5]):
            response += f"✈️ {flight['Airline']}\n"
            response += f"   - From: {flight.get('Departure Airport Code', 'N/A')}\n"
            response += f"   - To: {flight.get('Arrival Airport Code', 'N/A')}\n"
            response += f"   - Price: ${flight.get('Total Fare (EUR)', 'N/A')}\n"
            response += f"   - Duration: {flight.get('Duration (hrs)', 'N/A')} hours\n"
            response += f"   - Class: {flight.get('Class', 'N/A')}\n\n"
        
        response += "Would you like to:\n"
        response += "1. See flights from a specific location\n"
        response += "2. Filter by airline or flight duration\n"
        response += "3. See business class options\n"
        response += "4. Get recommendations for a different destination"
        
        return response, [represent_flight(f) for f in matching_flights[:5]]
        
    elif query_type == "city":
        matching_cities = []
        for city in data["cities"]:
            matching_cities.append(city)
        
        response = "Here are some great travel destinations:\n\n"
        for i, city in enumerate(matching_cities[:5]):
            response += f"🌆 {city['City']}, {city['Country']}\n"
            response += f"   - Avg. apartment price: ${city.get('Accommodation: Average price of 1 bedroom apartment per month', 'N/A')}/month\n"
            response += f"   - Avg. meal cost: ${city.get('Food: Average cost of a meal at a local, mid-level restaurant', 'N/A')}\n"
            response += f"   - WiFi speed: {city.get('Remote connection: Average WiFi speed (Mbps per second)', 'N/A')} Mbps\n"
            response += f"   - Attractions: {city.get('Tourist attractions: Number of Things to do on Tripadvisor', 'N/A')} things to do\n\n"
        
        response += "Would you like more details about:\n"
        response += "1. Digital nomad-friendly cities\n"
        response += "2. Budget travel destinations\n"
        response += "3. Luxury travel options\n"
        response += "4. A specific city"
        
        return response, [represent_city(c) for c in matching_cities[:5]]
        
    elif query_type == "complete_trip":
        # Extract destination from query
        destination = None
        duration = 5  # default
        
        # Check for duration in query
        duration_words = ["day", "week", "month"]
        for word in duration_words:
            if word in query.lower():
                try:
                    duration = int(query.lower().split(word)[0].split()[-1])
                    if word == "week":
                        duration *= 7
                    elif word == "month":
                        duration *= 30
                except:
                    pass
        
        for city in data["cities"]:
            if city['City'].lower() in query.lower():
                destination = city
                break
        
        if not destination:
            return "Please specify a destination city for your trip plan (e.g., 'Plan a 5-day trip to Paris').", []
        
        # Get relevant items
        city_hotels = [h for h in data["hotels"] if h.get("City", "").lower() == destination['City'].lower()]
        city_restaurants = [r for r in data["restaurants"] if r.get("City", "").lower() == destination['City'].lower()]
        city_flights = [f for f in data["flights"] if f.get("Arrival Airport Code", "").lower() == destination['City'].lower()]
        
        # Build itinerary
        attractions = destination.get('Tourist attractions: Number of Things to do on Tripadvisor', 'many')
        
        response = f"Here's a suggested {duration}-day itinerary for {destination['City']}, {destination['Country']}:\n\n"
        
        # Day 1: Arrival
        response += "📅 Day 1: Arrival & First Impressions\n"
        if city_flights:
            cheapest_flight = min(city_flights, key=lambda x: float(x.get("Total Fare (EUR)", 99999)))
            response += f"✈️ Flight: {cheapest_flight['Airline']} from {cheapest_flight['Departure Airport Code']} for ${cheapest_flight['Total Fare (EUR)']} ({cheapest_flight['Duration (hrs)']} hrs)\n"
        if city_hotels:
            mid_range_hotel = sorted(city_hotels, key=lambda x: float(x.get("price", 0)))[len(city_hotels)//2]
            response += f"🏨 Hotel: {mid_range_hotel['name']} (${mid_range_hotel['price']}/night, {mid_range_hotel.get('rating', 'N/A')}★)\n"
            response += f"   - Address: {mid_range_hotel.get('address', 'N/A')}\n"
        response += "   - After checking in, take a walk around the neighborhood to get oriented\n"
        if city_restaurants:
            local_restaurant = city_restaurants[0]
            response += f"🍽️ Dinner: {local_restaurant['Restaurant Name']} ({local_restaurant['Cuisines']}, ${local_restaurant['Average Cost for two']} for two)\n"
            response += f"   - Rating: {local_restaurant.get('Aggregate rating', 'N/A')}★\n\n"
        
        # Day 2: Sightseeing
        response += f"📅 Day 2: Explore {destination['City']}\n"
        response += "   - Morning: Visit top historical attractions (suggested: main landmarks)\n"
        response += "   - Afternoon: Take a guided walking tour or explore local markets\n"
        response += "   - Evening: Enjoy local entertainment or nightlife\n"
        if len(city_restaurants) > 1:
            response += f"🍽️ Dinner: {city_restaurants[1]['Restaurant Name']} ({city_restaurants[1]['Cuisines']}, ${city_restaurants[1]['Average Cost for two']} for two)\n\n"
        
        # Day 3: Cultural Experiences
        response += f"📅 Day 3: Cultural Immersion\n"
        response += "   - Morning: Visit museums or cultural centers\n"
        response += "   - Afternoon: Take a cooking class or craft workshop\n"
        response += "   - Evening: Attend a traditional performance\n\n"
        
        # Day 4: Day Trip
        response += f"📅 Day 4: Day Trip\n"
        response += "   - Full-day excursion to nearby attractions\n"
        response += "   - Suggested: Famous nearby sites or natural wonders\n\n"
        
        # Day 5: Relaxation & Departure
        response += f"📅 Day 5: Relaxation & Departure\n"
        response += "   - Morning: Last-minute shopping or visit favorite spots\n"
        response += "   - Afternoon: Check out from hotel\n"
        if city_flights:
            response += f"✈️ Flight: {cheapest_flight['Airline']} to {cheapest_flight['Departure Airport Code']}\n\n"
        
        # Budget estimate
        total_cost = 0
        if city_flights:
            total_cost += float(cheapest_flight['Total Fare (EUR)']) * 2  # round trip
        if city_hotels:
            total_cost += float(mid_range_hotel['price']) * duration
        if city_restaurants:
            total_cost += float(city_restaurants[0]['Average Cost for two']) * duration / 2
        # Add activities estimate
        total_cost += 50 * duration  # approx $50/day for activities
        
        response += f"💰 Estimated total cost for this trip: ${total_cost:.2f} (for one person)\n\n"
        
        response += "Would you like me to:\n"
        response += "1. Adjust this itinerary (higher/lower budget)\n"
        response += "2. Focus on specific interests (culture, food, adventure)\n"
        response += "3. Provide more detailed daily activities\n"
        response += "4. Book any of these options"
        
        retrieved = []
        if city_flights: retrieved.append(represent_flight(cheapest_flight))
        if city_hotels: retrieved.append(represent_hotel(mid_range_hotel))
        if city_restaurants: retrieved.extend([represent_restaurant(r) for r in city_restaurants[:2]])
        retrieved.append(represent_city(destination))
        
        return response, retrieved
        
    else:
        # For general queries, use the generator with a better prompt
        prompt = f"""You are a knowledgeable travel assistant. Provide a helpful, detailed response to this travel question:

Question: {query}

Available information:
{retrieved_docs}

Response:"""
        
        result = generator(prompt, num_return_sequences=1)
        response = result[0]["generated_text"].replace(prompt, "").strip()
        
        return response, retrieved_docs

# ----------------------------
# 7. Flask Web Interface (unchanged)
# ----------------------------
HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>Enhanced Travel Assistant</title>
    <style>
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            max-width: 1200px;
            margin: 0 auto;
            padding: 20px;
            background-color: #f5f7fa;
            color: #333;
        }
        .header {
            background-color: #4285f4;
            color: white;
            padding: 20px;
            border-radius: 8px;
            margin-bottom: 20px;
            text-align: center;
        }
        .filter-section {
            background-color: white;
            padding: 15px;
            border-radius: 8px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        .filter-buttons {
            display: flex;
            gap: 10px;
            flex-wrap: wrap;
            margin-top: 10px;
        }
        .filter-button {
            padding: 8px 15px;
            background-color: #e0e0e0;
            border: none;
            border-radius: 20px;
            cursor: pointer;
            transition: background-color 0.3s;
        }
        .filter-button:hover {
            background-color: #d0d0d0;
        }
        .filter-button.active {
            background-color: #4285f4;
            color: white;
        }
        .chat-container {
            background-color: white;
            border-radius: 8px;
            padding: 20px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            height: 500px;
            overflow-y: auto;
            white-space: pre-wrap;
        }
        .message {
            margin-bottom: 15px;
            padding: 10px 15px;
            border-radius: 18px;
            max-width: 80%;
            word-wrap: break-word;
        }
        .user-message {
            background-color: #e3f2fd;
            margin-left: auto;
            border-bottom-right-radius: 4px;
        }
        .bot-message {
            background-color: #f1f1f1;
            margin-right: auto;
            border-bottom-left-radius: 4px;
            white-space: pre-wrap;
        }
        .data-section {
            background-color: white;
            border-radius: 8px;
            padding: 20px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            max-height: 300px;
            overflow-y: auto;
        }
        .data-item {
            padding: 10px;
            border-bottom: 1px solid #eee;
            font-family: monospace;
        }
        .input-section {
            display: flex;
            gap: 10px;
        }
        #user-input {
            flex-grow: 1;
            padding: 12px;
            border: 1px solid #ddd;
            border-radius: 8px;
            font-size: 16px;
        }
        #submit-button {
            padding: 12px 20px;
            background-color: #4285f4;
            color: white;
            border: none;
            border-radius: 8px;
            cursor: pointer;
            font-size: 16px;
        }
        #submit-button:hover {
            background-color: #3367d6;
        }
        .query-type-indicator {
            font-size: 14px;
            color: #666;
            margin-top: 5px;
            font-style: italic;
        }
        .clear-button {
            padding: 8px 15px;
            background-color: #f44336;
            color: white;
            border: none;
            border-radius: 8px;
            cursor: pointer;
            font-size: 14px;
            margin-top: 10px;
        }
        .clear-button:hover {
            background-color: #d32f2f;
        }
    </style>
</head>
<body>
    <div class="header">
        <h1>Enhanced Travel Assistant</h1>
        <p>Get personalized travel recommendations for flights, hotels, restaurants and destinations</p>
    </div>
    
    <div class="filter-section">
        <h3>Not sure what to ask? Try these:</h3>
        <div class="filter-buttons">
            <button class="filter-button" onclick="setQuery('Best hotels in New York under $200')">Hotels</button>
            <button class="filter-button" onclick="setQuery('Italian restaurants in London')">Restaurants</button>
            <button class="filter-button" onclick="setQuery('Cheapest flights to Dubai next month')">Flights</button>
            <button class="filter-button" onclick="setQuery('Best digital nomad cities with good WiFi')">Destinations</button>
            <button class="filter-button" onclick="setQuery('Plan a complete 5-day trip to Istanbul')">Complete Trip</button>
        </div>
        <button class="clear-button" onclick="setQuery('clear history')">Clear Conversation</button>
    </div>
    
    <div class="chat-container" id="chat-container">
        {% for msg in history %}
            <div class="message {% if msg.sender == 'User' %}user-message{% else %}bot-message{% endif %}">
                <strong>{{ msg.sender }}:</strong> {{ msg.text }}
                {% if msg.query_type %}
                <div class="query-type-indicator">Detected as: {{ msg.query_type }}</div>
                {% endif %}
            </div>
        {% endfor %}
    </div>
    
    <div class="data-section">
        <h3>Raw Data Details</h3>
        {% if retrieved_data %}
            {% for doc in retrieved_data %}
                <div class="data-item">{{ doc }}</div>
            {% endfor %}
        {% else %}
            <div class="data-item">No raw data retrieved yet. Ask about hotels, restaurants, flights or destinations.</div>
        {% endif %}
    </div>
    
    <form method="post" class="input-section">
        <input type="text" id="user-input" name="question" placeholder="Ask about hotels, flights, restaurants or destinations..." required>
        <input type="submit" id="submit-button" value="Send">
    </form>
    
    <script>
        function setQuery(query) {
            document.getElementById('user-input').value = query;
            if (query.toLowerCase().includes('clear')) {
                document.forms[0].submit();
            }
            document.getElementById('user-input').focus();
        }
        
        // Auto-scroll chat to bottom
        var chatContainer = document.getElementById("chat-container");
        chatContainer.scrollTop = chatContainer.scrollHeight;
    </script>
</body>
</html>
"""

@app.route("/", methods=["GET", "POST"])
def index():
    global conversation_history
    retrieved_data = []
    
    if request.method == "POST":
        question = request.form["question"]
        query_type = detect_query_type(question)
        conversation_history.append({
            "sender": "User", 
            "text": question,
            "query_type": query_type.replace("_", " ").title()
        })
        
        answer, retrieved_data = generate_response(question)
        conversation_history.append({
            "sender": "Assistant", 
            "text": answer
        })
    
    return render_template_string(HTML_TEMPLATE, history=conversation_history, retrieved_data=retrieved_data)

# ----------------------------
# 8. Run the Application
# ----------------------------
if __name__ == "__main__":
    app.run(port=5001, debug=True)

C:\Users\mihab\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
C:\Users\mihab\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data loaded successfully!
Total representations for retrieval: 12314
Computing embeddings...


Device set to use cpu


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

C:\Users\mihab\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\IPython\core\interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# 4


In [ ]:
# Cell 1: Setup and Imports
import ast
import threading
from flask import Flask, request, render_template_string
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

app = Flask(__name__)

# CSV file paths (all in the adjusted_datasets folder)
CITIES_CSV = "adjusted_datasets/adjusted_cities.csv"
FLIGHTS_CSV = "adjusted_datasets/adjusted_flights.csv"
HOTELS_CSV = "adjusted_datasets/adjusted_hotels.csv"
RESTAURANTS_CSV = "adjusted_datasets/adjusted_restaurants.csv"
PREFERENCES_CSV = "adjusted_datasets/preferences.csv"
USERS_CSV = "adjusted_datasets/users.csv"
PASSPORTS_CSV = "adjusted_datasets/adjusted_passports.csv"
HISTORIES_CSV = "adjusted_datasets/histories.csv"

# Global conversation history for the chat interface
conversation_history = []

# ----------------------------
# 2. Data Loading (without Neo4j)
# ----------------------------
def load_data():
    # Load all CSV files into DataFrames
    cities_df = pd.read_csv(CITIES_CSV)
    flights_df = pd.read_csv(FLIGHTS_CSV)
    hotels_df = pd.read_csv(HOTELS_CSV)
    restaurants_df = pd.read_csv(RESTAURANTS_CSV)
    preferences_df = pd.read_csv(PREFERENCES_CSV)
    users_df = pd.read_csv(USERS_CSV)
    passports_df = pd.read_csv(PASSPORTS_CSV)
    histories_df = pd.read_csv(HISTORIES_CSV)
    
    # Convert DataFrames to dictionaries for easier access
    data = {
        "cities": cities_df.to_dict('records'),
        "flights": flights_df.to_dict('records'),
        "hotels": hotels_df.to_dict('records'),
        "restaurants": restaurants_df.to_dict('records'),
        "preferences": preferences_df.to_dict('records'),
        "users": users_df.to_dict('records'),
        "passports": passports_df.to_dict('records'),
        "histories": histories_df.to_dict('records')
    }
    return data

# Load all data
data = load_data()
print("Data loaded successfully!")

# ----------------------------
# 3. Representation and Retrieval
# ----------------------------
def build_representation(item, fields):
    parts = []
    for field, label in fields.items():
        value = item.get(field)
        if value is not None and str(value).strip() != "":
            parts.append(f"{label}: {value}")
    return "; ".join(parts)

def represent_city(city):
    fields = {
        "City": "City", "Country": "Country",
        "Remote connection: Average WiFi speed (Mbps per second)": "WiFi Speed",
        "Co-working spaces: Number of co-working spaces": "Co-working Spaces",
        "Accommodation: Average price of 1 bedroom apartment per month": "Apartment Price",
        "Food: Average cost of a meal at a local, mid-level restaurant": "Meal Cost",
        "Tourist attractions: Number of Things to do on Tripadvisor": "Attractions"
    }
    return build_representation(city, fields)

def represent_flight(flight):
    fields = {
        "Airline": "Airline", "Total Fare (EUR)": "Price",
        "Departure Airport Code": "From", "Arrival Airport Code": "To",
        "Duration (hrs)": "Duration", "Class": "Class"
    }
    return build_representation(flight, fields)

def represent_hotel(hotel):
    fields = {
        "name": "Name", "price": "Price",
        "number_reviews": "Reviews", "City": "City",
        "rating": "Rating", "address": "Address"
    }
    return build_representation(hotel, fields)

def represent_restaurant(restaurant):
    fields = {
        "Restaurant Name": "Name", "Cuisines": "Cuisines",
        "Average Cost for two": "Price for Two", "City": "City",
        "Aggregate rating": "Rating", "Address": "Address"
    }
    return build_representation(restaurant, fields)

# Create representations for all data
representations = []
for city in data["cities"]:
    representations.append(represent_city(city))
for flight in data["flights"]:
    representations.append(represent_flight(flight))
for hotel in data["hotels"]:
    representations.append(represent_hotel(hotel))
for restaurant in data["restaurants"]:
    representations.append(represent_restaurant(restaurant))

representations = list(set(representations))
print("Total representations for retrieval:", len(representations))

# ----------------------------
# 4. Embeddings and Retrieval
# ----------------------------
print("Computing embeddings...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embedder.encode(representations, convert_to_tensor=True)

def retrieve_documents(query, top_k=8, similarity_threshold=0.0):
    query_embedding = embedder.encode([query], convert_to_tensor=True)
    cos_scores = cosine_similarity(query_embedding.cpu().numpy(), doc_embeddings.cpu().numpy())[0]
    sorted_indices = np.argsort(cos_scores)[::-1]
    retrieved_docs = [representations[i] for i in sorted_indices[:top_k]]
    return retrieved_docs

# ----------------------------
# 5. Query Processing and Generation
# ----------------------------
generator = pipeline(
    "text-generation",
    model="gpt2",
    do_sample=True,
    temperature=0.7,
    max_new_tokens=200,
    no_repeat_ngram_size=3,
    repetition_penalty=1.2
)

def detect_query_type(query):
    query_lower = query.lower()
    if any(word in query_lower for word in ["hotel", "stay", "accommodation", "lodging"]):
        return "hotel"
    elif any(word in query_lower for word in ["restaurant", "eat", "dine", "food", "cuisine"]):
        return "restaurant"
    elif any(word in query_lower for word in ["flight", "fly", "airline", "ticket"]):
        return "flight"
    elif any(word in query_lower for word in ["city", "destination", "place", "visit", "location"]):
        return "city"
    elif any(word in query_lower for word in ["trip", "itinerary", "plan", "vacation", "holiday"]):
        return "complete_trip"
    elif any(word in query_lower for word in ["clear", "reset", "delete", "erase"]):
        return "clear_history"
    else:
        return "general"

def refine_query(raw_query):
    query_type = detect_query_type(raw_query)
    if query_type == "clear_history":
        return raw_query, query_type
        
    prompt = f"""
    Refine this travel query to be more specific for a {query_type} search:
    Original Query: {raw_query}
    Refined Query:"""
    result = generator(prompt, num_return_sequences=1)
    refined = result[0]["generated_text"].replace(prompt, "").strip().split("\n")[0].strip()
    return refined, query_type

# ----------------------------
# 6. IMPROVED Response Generation
# ----------------------------
def generate_response(query):
    # First detect what kind of information the user wants
    refined_query, query_type = refine_query(query)
    print(f"Detected query type: {query_type}, Refined: {refined_query}")
    
    # Handle clear history command
    if query_type == "clear_history":
        global conversation_history
        conversation_history = []
        return "I've cleared our conversation history. How can I help you with your travel plans?", []
    
    # Retrieve relevant documents based on query type
    retrieved_docs = retrieve_documents(refined_query, top_k=10)
    
    if not retrieved_docs:
        return "I couldn't find enough information about that. Could you be more specific?", []
    
    # Generate a prompt based on query type
    if query_type == "hotel":
        # Find hotels matching the query (e.g., price range)
        target_city = None
        max_price = None
        if "new york" in query.lower():
            target_city = "New York"
        if "under" in query.lower() and "$" in query.lower():
            try:
                max_price = float(query.split("$")[1].split()[0])
            except:
                pass
        
        matching_hotels = []
        for hotel in data["hotels"]:
            if target_city and hotel.get("City", "").lower() != target_city.lower():
                continue
            try:
                hotel_price = float(hotel.get("price", 99999))
                if max_price and hotel_price > max_price:
                    continue
                matching_hotels.append(hotel)
            except:
                continue
        
        if not matching_hotels:
            # Find cheapest hotel if none match the price
            if target_city:
                city_hotels = [h for h in data["hotels"] if h.get("City", "").lower() == target_city.lower()]
                if city_hotels:
                    try:
                        cheapest = min(city_hotels, key=lambda x: float(x.get("price", 99999)))
                        response = f"I couldn't find hotels under ${max_price} in {target_city}. The cheapest option available is:\n\n🏨 {cheapest['name']}\n   - Price: ${cheapest['price']}\n   - Reviews: {cheapest.get('number_reviews', 'N/A')}\n   - Rating: {cheapest.get('rating', 'N/A')}\n   - Address: {cheapest.get('address', 'N/A')}\n\nWould you like more information about this or other options?"
                        return response, [represent_hotel(cheapest)]
                    except:
                        pass
            
            return f"I couldn't find any hotels matching your criteria in our database. Please try a different search.", []
        
        # Sort by price then rating
        matching_hotels.sort(key=lambda x: (float(x.get("price", 99999)), -float(x.get("rating", 0))))
        
        # Build detailed response
        response = f"Here are the best hotel options in {target_city if target_city else 'our database'} under ${max_price if max_price else 'any price'}:\n\n"
        for i, hotel in enumerate(matching_hotels[:5]):  # Show top 5
            response += f"🏨 {hotel['name']}\n"
            response += f"   - Price: ${hotel['price']}\n"
            response += f"   - Reviews: {hotel.get('number_reviews', 'N/A')}\n"
            response += f"   - Rating: {hotel.get('rating', 'N/A')}/5\n"
            response += f"   - Address: {hotel.get('address', 'N/A')}\n\n"
        
        response += "Would you like:\n"
        response += "1. More details about any of these hotels\n"
        response += "2. Cheaper options in a different area\n"
        response += "3. Higher-end options with better amenities\n"
        response += "4. Something else?"
        
        return response, [represent_hotel(h) for h in matching_hotels[:5]]
        
    elif query_type == "restaurant":
        # Find restaurants matching the query
        target_city = None
        cuisine_type = None
        
        # Extract city if mentioned
        for city in data["cities"]:
            if city['City'].lower() in query.lower():
                target_city = city['City']
                break
                
        # Extract cuisine type if mentioned
        cuisine_words = ["italian", "chinese", "french", "japanese", "mexican", "indian", "thai"]
        for word in cuisine_words:
            if word in query.lower():
                cuisine_type = word
                break
        
        matching_restaurants = []
        for restaurant in data["restaurants"]:
            if target_city and restaurant.get("City", "").lower() != target_city.lower():
                continue
            if cuisine_type and cuisine_type not in restaurant.get("Cuisines", "").lower():
                continue
            matching_restaurants.append(restaurant)
        
        if not matching_restaurants:
            return f"I couldn't find any {cuisine_type + ' ' if cuisine_type else ''}restaurants matching your criteria in {target_city if target_city else 'our database'}. Please try a different search.", []
        
        # Sort by rating then price
        matching_restaurants.sort(key=lambda x: (-float(x.get("Aggregate rating", 0)), float(x.get("Average Cost for two", 0))))
        
        response = f"Here are some excellent {cuisine_type if cuisine_type else ''} restaurant options in {target_city if target_city else 'various cities'}:\n\n"
        for i, restaurant in enumerate(matching_restaurants[:5]):
            response += f"🍽️ {restaurant['Restaurant Name']}\n"
            response += f"   - Cuisine: {restaurant.get('Cuisines', 'N/A')}\n"
            response += f"   - Avg. cost for two: ${restaurant.get('Average Cost for two', 'N/A')}\n"
            response += f"   - Rating: {restaurant.get('Aggregate rating', 'N/A')}/5\n"
            response += f"   - Address: {restaurant.get('Address', 'N/A')}\n\n"
        
        response += "Would you like to:\n"
        response += "1. Filter by a specific price range\n"
        response += "2. See options in a different area\n"
        response += "3. Get recommendations for a different cuisine\n"
        response += "4. More details about any of these"
        
        return response, [represent_restaurant(r) for r in matching_restaurants[:5]]
        
    elif query_type == "flight":
        # Find flights matching the query
        target_destination = None
        max_price = None
        
        # Extract destination if mentioned
        for city in data["cities"]:
            if city['City'].lower() in query.lower():
                target_destination = city['City']
                break
                
        # Extract max price if mentioned
        if "under" in query.lower() and "$" in query.lower():
            try:
                max_price = float(query.split("$")[1].split()[0])
            except:
                pass
        
        matching_flights = []
        for flight in data["flights"]:
            if target_destination and flight.get("Arrival Airport Code", "").lower() != target_destination.lower():
                continue
            try:
                flight_price = float(flight.get("Total Fare (EUR)", 99999))
                if max_price and flight_price > max_price:
                    continue
                matching_flights.append(flight)
            except:
                continue
        
        if not matching_flights:
            return f"I couldn't find any flights matching your criteria. Please try a different search.", []
        
        # Sort by price
        matching_flights.sort(key=lambda x: float(x.get("Total Fare (EUR)", 99999)))
        
        response = f"Here are the best flight options to {target_destination if target_destination else 'various destinations'}:\n\n"
        for i, flight in enumerate(matching_flights[:5]):
            response += f"✈️ {flight['Airline']}\n"
            response += f"   - From: {flight.get('Departure Airport Code', 'N/A')}\n"
            response += f"   - To: {flight.get('Arrival Airport Code', 'N/A')}\n"
            response += f"   - Price: ${flight.get('Total Fare (EUR)', 'N/A')}\n"
            response += f"   - Duration: {flight.get('Duration (hrs)', 'N/A')} hours\n"
            response += f"   - Class: {flight.get('Class', 'N/A')}\n\n"
        
        response += "Would you like to:\n"
        response += "1. See flights from a specific location\n"
        response += "2. Filter by airline or flight duration\n"
        response += "3. See business class options\n"
        response += "4. Get recommendations for a different destination"
        
        return response, [represent_flight(f) for f in matching_flights[:5]]
        
    elif query_type == "city":
        matching_cities = []
        for city in data["cities"]:
            matching_cities.append(city)
        
        response = "Here are some great travel destinations:\n\n"
        for i, city in enumerate(matching_cities[:5]):
            response += f"🌆 {city['City']}, {city['Country']}\n"
            response += f"   - Avg. apartment price: ${city.get('Accommodation: Average price of 1 bedroom apartment per month', 'N/A')}/month\n"
            response += f"   - Avg. meal cost: ${city.get('Food: Average cost of a meal at a local, mid-level restaurant', 'N/A')}\n"
            response += f"   - WiFi speed: {city.get('Remote connection: Average WiFi speed (Mbps per second)', 'N/A')} Mbps\n"
            response += f"   - Attractions: {city.get('Tourist attractions: Number of Things to do on Tripadvisor', 'N/A')} things to do\n\n"
        
        response += "Would you like more details about:\n"
        response += "1. Digital nomad-friendly cities\n"
        response += "2. Budget travel destinations\n"
        response += "3. Luxury travel options\n"
        response += "4. A specific city"
        
        return response, [represent_city(c) for c in matching_cities[:5]]
        
    elif query_type == "complete_trip":
        # Extract destination from query
        destination = None
        duration = 5  # default
        
        # Check for duration in query
        duration_words = ["day", "week", "month"]
        for word in duration_words:
            if word in query.lower():
                try:
                    duration = int(query.lower().split(word)[0].split()[-1])
                    if word == "week":
                        duration *= 7
                    elif word == "month":
                        duration *= 30
                except:
                    pass
        
        for city in data["cities"]:
            if city['City'].lower() in query.lower():
                destination = city
                break
        
        if not destination:
            return "Please specify a destination city for your trip plan (e.g., 'Plan a 5-day trip to Paris').", []
        
        # Get relevant items
        city_hotels = [h for h in data["hotels"] if h.get("City", "").lower() == destination['City'].lower()]
        city_restaurants = [r for r in data["restaurants"] if r.get("City", "").lower() == destination['City'].lower()]
        city_flights = [f for f in data["flights"] if f.get("Arrival Airport Code", "").lower() == destination['City'].lower()]
        
        # Build detailed itinerary
        response = f"Here's a detailed {duration}-day itinerary for {destination['City']}, {destination['Country']}:\n\n"
        
        # Day 1: Arrival
        response += "📅 **Day 1: Arrival & First Impressions**\n"
        if city_flights:
            cheapest_flight = min(city_flights, key=lambda x: float(x.get("Total Fare (EUR)", 99999)))
            response += f"- ✈️ **Flight**: {cheapest_flight['Airline']} from {cheapest_flight['Departure Airport Code']} for ${cheapest_flight['Total Fare (EUR)']} ({cheapest_flight['Duration (hrs)']} hrs)\n"
        if city_hotels:
            mid_range_hotel = sorted(city_hotels, key=lambda x: float(x.get("price", 0)))[len(city_hotels)//2]
            response += f"- 🏨 **Hotel Check-in**: {mid_range_hotel['name']} (${mid_range_hotel['price']}/night, ★{mid_range_hotel.get('rating', 'N/A')} rating)\n"
            response += f"  - Address: {mid_range_hotel.get('address', 'N/A')}\n"
        response += "- 🚶 **Afternoon**: Explore the neighborhood around your hotel\n"
        if city_restaurants:
            local_restaurant = city_restaurants[0]
            response += f"- 🍽️ **Dinner**: {local_restaurant['Restaurant Name']} ({local_restaurant['Cuisines']}, ★{local_restaurant.get('Aggregate rating', 'N/A')} rating)\n"
            response += f"  - Avg. cost for two: ${local_restaurant['Average Cost for two']}\n"
            response += f"  - Address: {local_restaurant.get('Address', 'N/A')}\n\n"
        
        # Day 2: Sightseeing
        response += f"📅 **Day 2: Exploring {destination['City']}'s Highlights**\n"
        response += f"- 🌅 **Morning**: Visit top attractions (there are {destination.get('Tourist attractions: Number of Things to do on Tripadvisor', 'many')} options available)\n"
        response += "- 🚶 **Afternoon**: Take a guided walking tour or explore independently\n"
        if len(city_restaurants) > 1:
            response += f"- 🍽️ **Dinner**: {city_restaurants[1]['Restaurant Name']} ({city_restaurants[1]['Cuisines']}, ★{city_restaurants[1].get('Aggregate rating', 'N/A')} rating)\n\n"
        
        # Day 3-4: Customizable
        response += f"📅 **Day 3-{duration-1}: Customizable Experiences**\n"
        response += "- 🏛️ **Option 1**: Visit museums and cultural sites\n"
        response += "- 🚤 **Option 2**: Take a boat tour or day trip\n"
        response += "- 🛍️ **Option 3**: Shopping at local markets\n"
        response += "- ☕ **Option 4**: Relax at local cafés\n\n"
        
        # Last day: Departure
        response += f"📅 **Day {duration}: Departure**\n"
        response += "- 🏨 **Hotel Check-out**\n"
        if city_flights:
            response += f"- ✈️ **Flight Home**: {cheapest_flight['Airline']} to {cheapest_flight['Departure Airport Code']}\n\n"
        
        # Budget estimate
        total_cost = 0
        if city_flights:
            total_cost += float(cheapest_flight['Total Fare (EUR)']) * 2  # round trip
        if city_hotels:
            total_cost += float(mid_range_hotel['price']) * duration
        if city_restaurants:
            total_cost += float(city_restaurants[0]['Average Cost for two']) * duration / 2
        
        response += f"💰 **Estimated Budget**: ${total_cost:.2f} (for one person, excluding shopping/souvenirs)\n\n"
        
        response += "**Would you like to:**\n"
        response += "1. Adjust this itinerary (higher/lower budget)\n"
        response += "2. Focus on specific interests (culture, food, adventure)\n"
        response += "3. Get more detailed daily activities\n"
        response += "4. Book any of these options"
        
        retrieved = []
        if city_flights: retrieved.append(represent_flight(cheapest_flight))
        if city_hotels: retrieved.append(represent_hotel(mid_range_hotel))
        if city_restaurants: retrieved.extend([represent_restaurant(r) for r in city_restaurants[:2]])
        retrieved.append(represent_city(destination))
        
        return response, retrieved
        
    else:
        # For general queries, use the generator with better prompting
        prompt = f"""Based on this travel data:
        {retrieved_docs}
        
        Provide a comprehensive, detailed answer to the travel query: {query}
        Include specific recommendations with prices, ratings, and addresses when available.
        Structure the response clearly with bullet points and emojis for better readability.
        If recommending options, list 3-5 choices with their key features.
        Always include practical details like prices, locations, and ratings."""
        
        result = generator(prompt, num_return_sequences=1)
        response = result[0]["generated_text"].replace(prompt, "").strip()
        
        return response, retrieved_docs

# ----------------------------
# 7. Enhanced Flask Web Interface (same as before)
# ----------------------------
HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>Enhanced Travel Assistant</title>
    <style>
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            max-width: 1200px;
            margin: 0 auto;
            padding: 20px;
            background-color: #f5f7fa;
            color: #333;
        }
        .header {
            background-color: #4285f4;
            color: white;
            padding: 20px;
            border-radius: 8px;
            margin-bottom: 20px;
            text-align: center;
        }
        .filter-section {
            background-color: white;
            padding: 15px;
            border-radius: 8px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        .filter-buttons {
            display: flex;
            gap: 10px;
            flex-wrap: wrap;
            margin-top: 10px;
        }
        .filter-button {
            padding: 8px 15px;
            background-color: #e0e0e0;
            border: none;
            border-radius: 20px;
            cursor: pointer;
            transition: background-color 0.3s;
        }
        .filter-button:hover {
            background-color: #d0d0d0;
        }
        .filter-button.active {
            background-color: #4285f4;
            color: white;
        }
        .chat-container {
            background-color: white;
            border-radius: 8px;
            padding: 20px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            height: 500px;
            overflow-y: auto;
            white-space: pre-wrap;
        }
        .message {
            margin-bottom: 15px;
            padding: 10px 15px;
            border-radius: 18px;
            max-width: 80%;
            word-wrap: break-word;
        }
        .user-message {
            background-color: #e3f2fd;
            margin-left: auto;
            border-bottom-right-radius: 4px;
        }
        .bot-message {
            background-color: #f1f1f1;
            margin-right: auto;
            border-bottom-left-radius: 4px;
            white-space: pre-wrap;
        }
        .data-section {
            background-color: white;
            border-radius: 8px;
            padding: 20px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            max-height: 300px;
            overflow-y: auto;
        }
        .data-item {
            padding: 10px;
            border-bottom: 1px solid #eee;
            font-family: monospace;
        }
        .input-section {
            display: flex;
            gap: 10px;
        }
        #user-input {
            flex-grow: 1;
            padding: 12px;
            border: 1px solid #ddd;
            border-radius: 8px;
            font-size: 16px;
        }
        #submit-button {
            padding: 12px 20px;
            background-color: #4285f4;
            color: white;
            border: none;
            border-radius: 8px;
            cursor: pointer;
            font-size: 16px;
        }
        #submit-button:hover {
            background-color: #3367d6;
        }
        .query-type-indicator {
            font-size: 14px;
            color: #666;
            margin-top: 5px;
            font-style: italic;
        }
        .clear-button {
            padding: 8px 15px;
            background-color: #f44336;
            color: white;
            border: none;
            border-radius: 8px;
            cursor: pointer;
            font-size: 14px;
            margin-top: 10px;
        }
        .clear-button:hover {
            background-color: #d32f2f;
        }
    </style>
</head>
<body>
    <div class="header">
        <h1>Enhanced Travel Assistant</h1>
        <p>Get personalized travel recommendations for flights, hotels, restaurants and destinations</p>
    </div>
    
    <div class="filter-section">
        <h3>Not sure what to ask? Try these:</h3>
        <div class="filter-buttons">
            <button class="filter-button" onclick="setQuery('Best hotels in New York under $200')">Hotels</button>
            <button class="filter-button" onclick="setQuery('Italian restaurants in London')">Restaurants</button>
            <button class="filter-button" onclick="setQuery('Cheapest flights to Dubai next month')">Flights</button>
            <button class="filter-button" onclick="setQuery('Best digital nomad cities with good WiFi')">Destinations</button>
            <button class="filter-button" onclick="setQuery('Plan a complete 5-day trip to Istanbul')">Complete Trip</button>
        </div>
        <button class="clear-button" onclick="setQuery('clear history')">Clear Conversation</button>
    </div>
    
    <div class="chat-container" id="chat-container">
        {% for msg in history %}
            <div class="message {% if msg.sender == 'User' %}user-message{% else %}bot-message{% endif %}">
                <strong>{{ msg.sender }}:</strong> {{ msg.text }}
                {% if msg.query_type %}
                <div class="query-type-indicator">Detected as: {{ msg.query_type }}</div>
                {% endif %}
            </div>
        {% endfor %}
    </div>
    
    <div class="data-section">
        <h3>Raw Data Details</h3>
        {% if retrieved_data %}
            {% for doc in retrieved_data %}
                <div class="data-item">{{ doc }}</div>
            {% endfor %}
        {% else %}
            <div class="data-item">No raw data retrieved yet. Ask about hotels, restaurants, flights or destinations.</div>
        {% endif %}
    </div>
    
    <form method="post" class="input-section">
        <input type="text" id="user-input" name="question" placeholder="Ask about hotels, flights, restaurants or destinations..." required>
        <input type="submit" id="submit-button" value="Send">
    </form>
    
    <script>
        function setQuery(query) {
            document.getElementById('user-input').value = query;
            if (query.toLowerCase().includes('clear')) {
                document.forms[0].submit();
            }
            document.getElementById('user-input').focus();
        }
        
        // Auto-scroll chat to bottom
        var chatContainer = document.getElementById("chat-container");
        chatContainer.scrollTop = chatContainer.scrollHeight;
    </script>
</body>
</html>
"""

@app.route("/", methods=["GET", "POST"])
def index():
    global conversation_history
    retrieved_data = []
    
    if request.method == "POST":
        question = request.form["question"]
        query_type = detect_query_type(question)
        conversation_history.append({
            "sender": "User", 
            "text": question,
            "query_type": query_type.replace("_", " ").title()
        })
        
        answer, retrieved_data = generate_response(question)
        conversation_history.append({
            "sender": "Assistant", 
            "text": answer
        })
    
    return render_template_string(HTML_TEMPLATE, history=conversation_history, retrieved_data=retrieved_data)

# ----------------------------
# 8. Run the Application
# ----------------------------
if __name__ == "__main__":
    app.run(port=5001, debug=True)

#  5 Code der angepasst ist


In [ ]:
# # Cell 1: Setup and Imports
# import ast
# import threading
# from flask import Flask, request, render_template_string
# import pandas as pd
# import numpy as np
# from sentence_transformers import SentenceTransformer
# from sklearn.metrics.pairwise import cosine_similarity
# from transformers import pipeline

# app = Flask(__name__)

# # CSV file paths (all in the adjusted_datasets folder)
# CITIES_CSV = "adjusted_datasets/adjusted_cities.csv"
# FLIGHTS_CSV = "adjusted_datasets/adjusted_flights.csv"
# HOTELS_CSV = "adjusted_datasets/adjusted_hotels.csv"
# RESTAURANTS_CSV = "adjusted_datasets/adjusted_restaurants.csv"
# PREFERENCES_CSV = "adjusted_datasets/preferences.csv"
# USERS_CSV = "adjusted_datasets/users.csv"
# PASSPORTS_CSV = "adjusted_datasets/adjusted_passports.csv"
# HISTORIES_CSV = "adjusted_datasets/histories.csv"

# # Global conversation history for the chat interface
# conversation_history = []

# # ----------------------------
# # 2. Data Loading (without Neo4j)
# # ----------------------------
# def load_data():
#     # Load all CSV files into DataFrames
#     cities_df = pd.read_csv(CITIES_CSV)
#     flights_df = pd.read_csv(FLIGHTS_CSV)
#     hotels_df = pd.read_csv(HOTELS_CSV)
#     restaurants_df = pd.read_csv(RESTAURANTS_CSV)
#     preferences_df = pd.read_csv(PREFERENCES_CSV)
#     users_df = pd.read_csv(USERS_CSV)
#     passports_df = pd.read_csv(PASSPORTS_CSV)
#     histories_df = pd.read_csv(HISTORIES_CSV)
    
#     # Convert DataFrames to dictionaries for easier access
#     data = {
#         "cities": cities_df.to_dict('records'),
#         "flights": flights_df.to_dict('records'),
#         "hotels": hotels_df.to_dict('records'),
#         "restaurants": restaurants_df.to_dict('records'),
#         "preferences": preferences_df.to_dict('records'),
#         "users": users_df.to_dict('records'),
#         "passports": passports_df.to_dict('records'),
#         "histories": histories_df.to_dict('records')
#     }
#     return data

# # Load all data
# data = load_data()
# print("Data loaded successfully!")

# # ----------------------------
# # 3. Representation and Retrieval
# # ----------------------------
# def build_representation(item, fields):
#     parts = []
#     for field, label in fields.items():
#         value = item.get(field)
#         if value is not None and str(value).strip() != "":
#             parts.append(f"{label}: {value}")
#     return "; ".join(parts)

# def represent_city(city):
#     fields = {
#         "City": "City", "Country": "Country",
#         "Remote connection: Average WiFi speed (Mbps per second)": "WiFi Speed",
#         "Co-working spaces: Number of co-working spaces": "Co-working Spaces",
#         "Accommodation: Average price of 1 bedroom apartment per month": "Apartment Price",
#         "Food: Average cost of a meal at a local, mid-level restaurant": "Meal Cost",
#         "Tourist attractions: Number of Things to do on Tripadvisor": "Attractions"
#     }
#     return build_representation(city, fields)

# def represent_flight(flight):
#     fields = {
#         "Airline": "Airline", "Total Fare (EUR)": "Price",
#         "Departure Airport Code": "From", "Arrival Airport Code": "To",
#         "Duration (hrs)": "Duration", "Class": "Class"
#     }
#     return build_representation(flight, fields)

# def represent_hotel(hotel):
#     fields = {
#         "name": "Name", "price": "Price",
#         "number_reviews": "Reviews", "City": "City"
#     }
#     return build_representation(hotel, fields)

# def represent_restaurant(restaurant):
#     fields = {
#         "Restaurant Name": "Name", "Cuisines": "Cuisines",
#         "Average Cost for two": "Price for Two", "City": "City"
#     }
#     return build_representation(restaurant, fields)

# # Create representations for all data
# representations = []
# for city in data["cities"]:
#     representations.append(represent_city(city))
# for flight in data["flights"]:
#     representations.append(represent_flight(flight))
# for hotel in data["hotels"]:
#     representations.append(represent_hotel(hotel))
# for restaurant in data["restaurants"]:
#     representations.append(represent_restaurant(restaurant))

# representations = list(set(representations))
# print("Total representations for retrieval:", len(representations))

# # ----------------------------
# # 4. Embeddings and Retrieval
# # ----------------------------
# print("Computing embeddings...")
# embedder = SentenceTransformer("all-MiniLM-L6-v2")
# doc_embeddings = embedder.encode(representations, convert_to_tensor=True)

# def retrieve_documents(query, top_k=8, similarity_threshold=0.0):
#     query_embedding = embedder.encode([query], convert_to_tensor=True)
#     cos_scores = cosine_similarity(query_embedding.cpu().numpy(), doc_embeddings.cpu().numpy())[0]
#     sorted_indices = np.argsort(cos_scores)[::-1]
#     retrieved_docs = [representations[i] for i in sorted_indices[:top_k]]
#     return retrieved_docs

# # ----------------------------
# # 5. Query Processing and Generation
# # ----------------------------
# generator = pipeline(
#     "text-generation",
#     model="gpt2",
#     do_sample=True,
#     temperature=0.7,
#     max_new_tokens=200,
#     no_repeat_ngram_size=3,
#     repetition_penalty=1.2
# )

# def detect_query_type(query):
#     query_lower = query.lower()
#     if any(word in query_lower for word in ["hotel", "stay", "accommodation", "lodging"]):
#         return "hotel"
#     elif any(word in query_lower for word in ["restaurant", "eat", "dine", "food", "cuisine"]):
#         return "restaurant"
#     elif any(word in query_lower for word in ["flight", "fly", "airline", "ticket"]):
#         return "flight"
#     elif any(word in query_lower for word in ["city", "destination", "place", "visit", "location"]):
#         return "city"
#     elif any(word in query_lower for word in ["trip", "itinerary", "plan", "vacation", "holiday"]):
#         return "complete_trip"
#     elif any(word in query_lower for word in ["clear", "reset", "delete", "erase"]):
#         return "clear_history"
#     else:
#         return "general"

# def refine_query(raw_query):
#     query_type = detect_query_type(raw_query)
#     if query_type == "clear_history":
#         return raw_query, query_type
        
#     prompt = f"""
#     Refine this travel query to be more specific for a {query_type} search:
#     Original Query: {raw_query}
#     Refined Query:"""
#     result = generator(prompt, num_return_sequences=1)
#     refined = result[0]["generated_text"].replace(prompt, "").strip().split("\n")[0].strip()
#     return refined, query_type

# # ----------------------------
# # 6. Response Generation (IMPROVED)
# # ----------------------------
# def generate_response(query):
#     # First detect what kind of information the user wants
#     refined_query, query_type = refine_query(query)
#     print(f"Detected query type: {query_type}, Refined: {refined_query}")
    
#     # Handle clear history command
#     if query_type == "clear_history":
#         global conversation_history
#         conversation_history = []
#         return "I've cleared our conversation history. How can I help you with your travel plans?", []
    
#     # Retrieve relevant documents based on query type
#     retrieved_docs = retrieve_documents(refined_query, top_k=10)
    
#     if not retrieved_docs:
#         return "I couldn't find enough information about that. Could you be more specific?", []
    
#     # Generate a prompt based on query type
#     if query_type == "hotel":
#         # Find hotels matching the query (e.g., price range)
#         target_city = None
#         max_price = None
#         if "new york" in query.lower():
#             target_city = "New York"
#         if "under" in query.lower() and "$" in query.lower():
#             try:
#                 max_price = float(query.split("$")[1].split()[0])
#             except:
#                 pass
        
#         matching_hotels = []
#         for hotel in data["hotels"]:
#             if target_city and hotel.get("City", "").lower() != target_city.lower():
#                 continue
#             try:
#                 hotel_price = float(hotel.get("price", 99999))
#                 if max_price and hotel_price > max_price:
#                     continue
#                 matching_hotels.append(hotel)
#             except:
#                 continue
        
#         if not matching_hotels:
#             # Find cheapest hotel if none match the price
#             if target_city:
#                 city_hotels = [h for h in data["hotels"] if h.get("City", "").lower() == target_city.lower()]
#                 if city_hotels:
#                     try:
#                         cheapest = min(city_hotels, key=lambda x: float(x.get("price", 99999)))
#                         response = f"I couldn't find hotels under ${max_price} in {target_city}. The cheapest option available is:\n\n🏨 {cheapest['name']}\n   - Price: ${cheapest['price']}\n   - Reviews: {cheapest.get('number_reviews', 'N/A')}\n\nWould you like more information about this or other options?"
#                         return response, [represent_hotel(cheapest)]
#                     except:
#                         pass
            
#             return f"I couldn't find any hotels matching your criteria in our database. Please try a different search.", []
        
#         # Sort by price
#         matching_hotels.sort(key=lambda x: float(x.get("price", 99999)))
        
#         # Build response
#         response = f"Here are the best hotel options in {target_city if target_city else 'our database'} under ${max_price if max_price else 'any price'}:\n\n"
#         for i, hotel in enumerate(matching_hotels[:5]):  # Show top 5
#             response += f"🏨 {hotel['name']}\n"
#             response += f"   - Price: ${hotel['price']}\n"
#             response += f"   - Reviews: {hotel.get('number_reviews', 'N/A')}\n"
#             response += f"   - City: {hotel.get('City', 'N/A')}\n\n"
        
#         response += "Would you like:\n"
#         response += "1. More details about any of these hotels\n"
#         response += "2. Cheaper options in a different area\n"
#         response += "3. Higher-end options with better amenities\n"
#         response += "4. Something else?"
        
#         return response, [represent_hotel(h) for h in matching_hotels[:5]]
        
#     elif query_type == "restaurant":
#         # Find restaurants matching the query
#         target_city = None
#         cuisine_type = None
        
#         # Extract city if mentioned
#         for city in data["cities"]:
#             if city['City'].lower() in query.lower():
#                 target_city = city['City']
#                 break
                
#         # Extract cuisine type if mentioned
#         cuisine_words = ["italian", "chinese", "french", "japanese", "mexican", "indian", "thai"]
#         for word in cuisine_words:
#             if word in query.lower():
#                 cuisine_type = word
#                 break
        
#         matching_restaurants = []
#         for restaurant in data["restaurants"]:
#             if target_city and restaurant.get("City", "").lower() != target_city.lower():
#                 continue
#             if cuisine_type and cuisine_type not in restaurant.get("Cuisines", "").lower():
#                 continue
#             matching_restaurants.append(restaurant)
        
#         if not matching_restaurants:
#             return f"I couldn't find any {cuisine_type + ' ' if cuisine_type else ''}restaurants matching your criteria in {target_city if target_city else 'our database'}. Please try a different search.", []
        
#         # Sort by price
#         matching_restaurants.sort(key=lambda x: float(x.get("Average Cost for two", 0)))
        
#         response = f"Here are some excellent {cuisine_type if cuisine_type else ''} restaurant options in {target_city if target_city else 'various cities'}:\n\n"
#         for i, restaurant in enumerate(matching_restaurants[:5]):
#             response += f"🍽️ {restaurant['Restaurant Name']}\n"
#             response += f"   - Cuisine: {restaurant.get('Cuisines', 'N/A')}\n"
#             response += f"   - Avg. cost for two: ${restaurant.get('Average Cost for two', 'N/A')}\n"
#             response += f"   - Location: {restaurant.get('City', 'N/A')}\n\n"
        
#         response += "Would you like to:\n"
#         response += "1. Filter by a specific price range\n"
#         response += "2. See options in a different area\n"
#         response += "3. Get recommendations for a different cuisine\n"
#         response += "4. More details about any of these"
        
#         return response, [represent_restaurant(r) for r in matching_restaurants[:5]]
        
#     elif query_type == "flight":
#         # Find flights matching the query
#         target_destination = None
#         max_price = None
        
#         # Extract destination if mentioned
#         for city in data["cities"]:
#             if city['City'].lower() in query.lower():
#                 target_destination = city['City']
#                 break
                
#         # Extract max price if mentioned
#         if "under" in query.lower() and "$" in query.lower():
#             try:
#                 max_price = float(query.split("$")[1].split()[0])
#             except:
#                 pass
        
#         matching_flights = []
#         for flight in data["flights"]:
#             if target_destination and flight.get("Arrival Airport Code", "").lower() != target_destination.lower():
#                 continue
#             try:
#                 flight_price = float(flight.get("Total Fare (EUR)", 99999))
#                 if max_price and flight_price > max_price:
#                     continue
#                 matching_flights.append(flight)
#             except:
#                 continue
        
#         if not matching_flights:
#             return f"I couldn't find any flights matching your criteria. Please try a different search.", []
        
#         # Sort by price
#         matching_flights.sort(key=lambda x: float(x.get("Total Fare (EUR)", 99999)))
        
#         response = f"Here are the best flight options to {target_destination if target_destination else 'various destinations'}:\n\n"
#         for i, flight in enumerate(matching_flights[:5]):
#             response += f"✈️ {flight['Airline']}\n"
#             response += f"   - From: {flight.get('Departure Airport Code', 'N/A')}\n"
#             response += f"   - To: {flight.get('Arrival Airport Code', 'N/A')}\n"
#             response += f"   - Price: ${flight.get('Total Fare (EUR)', 'N/A')}\n"
#             response += f"   - Duration: {flight.get('Duration (hrs)', 'N/A')} hours\n"
#             response += f"   - Class: {flight.get('Class', 'N/A')}\n\n"
        
#         response += "Would you like to:\n"
#         response += "1. See flights from a specific location\n"
#         response += "2. Filter by airline or flight duration\n"
#         response += "3. See business class options\n"
#         response += "4. Get recommendations for a different destination"
        
#         return response, [represent_flight(f) for f in matching_flights[:5]]
        
#     elif query_type == "city":
#         matching_cities = []
#         for city in data["cities"]:
#             matching_cities.append(city)
        
#         response = "Here are some great travel destinations:\n\n"
#         for i, city in enumerate(matching_cities[:5]):
#             response += f"🌆 {city['City']}, {city['Country']}\n"
#             response += f"   - Avg. apartment price: ${city.get('Accommodation: Average price of 1 bedroom apartment per month', 'N/A')}/month\n"
#             response += f"   - Avg. meal cost: ${city.get('Food: Average cost of a meal at a local, mid-level restaurant', 'N/A')}\n"
#             response += f"   - WiFi speed: {city.get('Remote connection: Average WiFi speed (Mbps per second)', 'N/A')} Mbps\n"
#             response += f"   - Attractions: {city.get('Tourist attractions: Number of Things to do on Tripadvisor', 'N/A')} things to do\n\n"
        
#         response += "Would you like more details about:\n"
#         response += "1. Digital nomad-friendly cities\n"
#         response += "2. Budget travel destinations\n"
#         response += "3. Luxury travel options\n"
#         response += "4. A specific city"
        
#         return response, [represent_city(c) for c in matching_cities[:5]]
        
#     elif query_type == "complete_trip":
#         # Extract destination from query
#         destination = None
#         duration = 5  # default
        
#         # Check for duration in query
#         duration_words = ["day", "week", "month"]
#         for word in duration_words:
#             if word in query.lower():
#                 try:
#                     duration = int(query.lower().split(word)[0].split()[-1])
#                     if word == "week":
#                         duration *= 7
#                     elif word == "month":
#                         duration *= 30
#                 except:
#                     pass
        
#         for city in data["cities"]:
#             if city['City'].lower() in query.lower():
#                 destination = city
#                 break
        
#         if not destination:
#             return "Please specify a destination city for your trip plan (e.g., 'Plan a 5-day trip to Paris').", []
        
#         # Get relevant items
#         city_hotels = [h for h in data["hotels"] if h.get("City", "").lower() == destination['City'].lower()]
#         city_restaurants = [r for r in data["restaurants"] if r.get("City", "").lower() == destination['City'].lower()]
#         city_flights = [f for f in data["flights"] if f.get("Arrival Airport Code", "").lower() == destination['City'].lower()]
        
#         # Build itinerary
#         attractions = destination.get('Tourist attractions: Number of Things to do on Tripadvisor', 'many')
        
#         response = f"Here's a suggested {duration}-day itinerary for {destination['City']}, {destination['Country']}:\n\n"
        
#         # Day 1: Arrival
#         response += "📅 Day 1: Arrival\n"
#         if city_flights:
#             cheapest_flight = min(city_flights, key=lambda x: float(x.get("Total Fare (EUR)", 99999)))
#             response += f"✈️ Flight: {cheapest_flight['Airline']} from {cheapest_flight['Departure Airport Code']} for ${cheapest_flight['Total Fare (EUR)']} ({cheapest_flight['Duration (hrs)']} hrs)\n"
#         if city_hotels:
#             mid_range_hotel = sorted(city_hotels, key=lambda x: float(x.get("price", 0)))[len(city_hotels)//2]
#             response += f"🏨 Hotel: {mid_range_hotel['name']} (${mid_range_hotel['price']}/night, {mid_range_hotel.get('number_reviews', 'N/A')} reviews)\n"
#         response += "   - Settle in and explore the neighborhood\n"
#         if city_restaurants:
#             local_restaurant = city_restaurants[0]
#             response += f"🍽️ Dinner: {local_restaurant['Restaurant Name']} ({local_restaurant['Cuisines']}, ${local_restaurant['Average Cost for two']} for two)\n\n"
        
#         # Day 2: Sightseeing
#         response += f"📅 Day 2: Explore {destination['City']}\n"
#         response += f"   - Visit top attractions ({attractions} options available)\n"
#         response += "   - Take a guided tour or explore on your own\n"
#         if len(city_restaurants) > 1:
#             response += f"🍽️ Dinner: {city_restaurants[1]['Restaurant Name']} ({city_restaurants[1]['Cuisines']})\n\n"
        
#         # Day 3-4: Customizable
#         response += f"📅 Day 3-{duration-1}: Customizable Activities\n"
#         response += "   - Cultural experiences\n"
#         response += "   - Day trips to nearby areas\n"
#         response += "   - Shopping and local markets\n"
#         response += "   - Relaxation time\n\n"
        
#         # Last day: Departure
#         response += f"📅 Day {duration}: Departure\n"
#         response += "   - Check out from hotel\n"
#         if city_flights:
#             response += f"✈️ Return flight: {cheapest_flight['Airline']} to {cheapest_flight['Departure Airport Code']}\n\n"
        
#         # Budget estimate
#         total_cost = 0
#         if city_flights:
#             total_cost += float(cheapest_flight['Total Fare (EUR)']) * 2  # round trip
#         if city_hotels:
#             total_cost += float(mid_range_hotel['price']) * duration
#         if city_restaurants:
#             total_cost += float(city_restaurants[0]['Average Cost for two']) * duration / 2
        
#         response += f"💰 Estimated total cost for this trip: ${total_cost:.2f} (for one person)\n\n"
        
#         response += "Would you like me to:\n"
#         response += "1. Adjust this itinerary (higher/lower budget)\n"
#         response += "2. Focus on specific interests (culture, food, adventure)\n"
#         response += "3. Provide more detailed daily activities\n"
#         response += "4. Book any of these options"
        
#         retrieved = []
#         if city_flights: retrieved.append(represent_flight(cheapest_flight))
#         if city_hotels: retrieved.append(represent_hotel(mid_range_hotel))
#         if city_restaurants: retrieved.extend([represent_restaurant(r) for r in city_restaurants[:2]])
#         retrieved.append(represent_city(destination))
        
#         return response, retrieved
        
#     else:
#         prompt = f"""Based on this travel data:
#         {retrieved_docs}
        
#         Provide a comprehensive travel answer to: {query}
#         Include relevant details about destinations, accommodations, and activities."""
        
#         result = generator(prompt, num_return_sequences=1)
#         response = result[0]["generated_text"].replace(prompt, "").strip()
        
#         return response, retrieved_docs

# # ----------------------------
# # 7. Enhanced Flask Web Interface
# # ----------------------------
# HTML_TEMPLATE = """
# <!DOCTYPE html>
# <html>
# <head>
#     <title>Enhanced Travel Assistant</title>
#     <style>
#         body {
#             font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
#             max-width: 1200px;
#             margin: 0 auto;
#             padding: 20px;
#             background-color: #f5f7fa;
#             color: #333;
#         }
#         .header {
#             background-color: #4285f4;
#             color: white;
#             padding: 20px;
#             border-radius: 8px;
#             margin-bottom: 20px;
#             text-align: center;
#         }
#         .filter-section {
#             background-color: white;
#             padding: 15px;
#             border-radius: 8px;
#             margin-bottom: 20px;
#             box-shadow: 0 2px 4px rgba(0,0,0,0.1);
#         }
#         .filter-buttons {
#             display: flex;
#             gap: 10px;
#             flex-wrap: wrap;
#             margin-top: 10px;
#         }
#         .filter-button {
#             padding: 8px 15px;
#             background-color: #e0e0e0;
#             border: none;
#             border-radius: 20px;
#             cursor: pointer;
#             transition: background-color 0.3s;
#         }
#         .filter-button:hover {
#             background-color: #d0d0d0;
#         }
#         .filter-button.active {
#             background-color: #4285f4;
#             color: white;
#         }
#         .chat-container {
#             background-color: white;
#             border-radius: 8px;
#             padding: 20px;
#             margin-bottom: 20px;
#             box-shadow: 0 2px 4px rgba(0,0,0,0.1);
#             height: 500px;
#             overflow-y: auto;
#             white-space: pre-wrap;
#         }
#         .message {
#             margin-bottom: 15px;
#             padding: 10px 15px;
#             border-radius: 18px;
#             max-width: 80%;
#             word-wrap: break-word;
#         }
#         .user-message {
#             background-color: #e3f2fd;
#             margin-left: auto;
#             border-bottom-right-radius: 4px;
#         }
#         .bot-message {
#             background-color: #f1f1f1;
#             margin-right: auto;
#             border-bottom-left-radius: 4px;
#             white-space: pre-wrap;
#         }
#         .data-section {
#             background-color: white;
#             border-radius: 8px;
#             padding: 20px;
#             margin-bottom: 20px;
#             box-shadow: 0 2px 4px rgba(0,0,0,0.1);
#             max-height: 300px;
#             overflow-y: auto;
#         }
#         .data-item {
#             padding: 10px;
#             border-bottom: 1px solid #eee;
#             font-family: monospace;
#         }
#         .input-section {
#             display: flex;
#             gap: 10px;
#         }
#         #user-input {
#             flex-grow: 1;
#             padding: 12px;
#             border: 1px solid #ddd;
#             border-radius: 8px;
#             font-size: 16px;
#         }
#         #submit-button {
#             padding: 12px 20px;
#             background-color: #4285f4;
#             color: white;
#             border: none;
#             border-radius: 8px;
#             cursor: pointer;
#             font-size: 16px;
#         }
#         #submit-button:hover {
#             background-color: #3367d6;
#         }
#         .query-type-indicator {
#             font-size: 14px;
#             color: #666;
#             margin-top: 5px;
#             font-style: italic;
#         }
#         .clear-button {
#             padding: 8px 15px;
#             background-color: #f44336;
#             color: white;
#             border: none;
#             border-radius: 8px;
#             cursor: pointer;
#             font-size: 14px;
#             margin-top: 10px;
#         }
#         .clear-button:hover {
#             background-color: #d32f2f;
#         }
#     </style>
# </head>
# <body>
#     <div class="header">
#         <h1>Enhanced Travel Assistant</h1>
#         <p>Get personalized travel recommendations for flights, hotels, restaurants and destinations</p>
#     </div>
    
#     <div class="filter-section">
#         <h3>Not sure what to ask? Try these:</h3>
#         <div class="filter-buttons">
#             <button class="filter-button" onclick="setQuery('Best hotels in New York under $200')">Hotels</button>
#             <button class="filter-button" onclick="setQuery('Italian restaurants in London')">Restaurants</button>
#             <button class="filter-button" onclick="setQuery('Cheapest flights to Dubai next month')">Flights</button>
#             <button class="filter-button" onclick="setQuery('Best digital nomad cities with good WiFi')">Destinations</button>
#             <button class="filter-button" onclick="setQuery('Plan a complete 5-day trip to Istanbul')">Complete Trip</button>
#         </div>
#         <button class="clear-button" onclick="setQuery('clear history')">Clear Conversation</button>
#     </div>
    
#     <div class="chat-container" id="chat-container">
#         {% for msg in history %}
#             <div class="message {% if msg.sender == 'User' %}user-message{% else %}bot-message{% endif %}">
#                 <strong>{{ msg.sender }}:</strong> {{ msg.text }}
#                 {% if msg.query_type %}
#                 <div class="query-type-indicator">Detected as: {{ msg.query_type }}</div>
#                 {% endif %}
#             </div>
#         {% endfor %}
#     </div>
    
#     <div class="data-section">
#         <h3>Raw Data Details</h3>
#         {% if retrieved_data %}
#             {% for doc in retrieved_data %}
#                 <div class="data-item">{{ doc }}</div>
#             {% endfor %}
#         {% else %}
#             <div class="data-item">No raw data retrieved yet. Ask about hotels, restaurants, flights or destinations.</div>
#         {% endif %}
#     </div>
    
#     <form method="post" class="input-section">
#         <input type="text" id="user-input" name="question" placeholder="Ask about hotels, flights, restaurants or destinations..." required>
#         <input type="submit" id="submit-button" value="Send">
#     </form>
    
#     <script>
#         function setQuery(query) {
#             document.getElementById('user-input').value = query;
#             if (query.toLowerCase().includes('clear')) {
#                 document.forms[0].submit();
#             }
#             document.getElementById('user-input').focus();
#         }
        
#         // Auto-scroll chat to bottom
#         var chatContainer = document.getElementById("chat-container");
#         chatContainer.scrollTop = chatContainer.scrollHeight;
#     </script>
# </body>
# </html>
# """

# @app.route("/", methods=["GET", "POST"])
# def index():
#     global conversation_history
#     retrieved_data = []
    
#     if request.method == "POST":
#         question = request.form["question"]
#         query_type = detect_query_type(question)
#         conversation_history.append({
#             "sender": "User", 
#             "text": question,
#             "query_type": query_type.replace("_", " ").title()
#         })
        
#         answer, retrieved_data = generate_response(question)
#         conversation_history.append({
#             "sender": "Assistant", 
#             "text": answer
#         })
    
#     return render_template_string(HTML_TEMPLATE, history=conversation_history, retrieved_data=retrieved_data)

# # ----------------------------
# # 8. Run the Application
# # ----------------------------
# if __name__ == "__main__":
#     app.run(port=5001, debug=True)

C:\Users\mihab\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
C:\Users\mihab\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data loaded successfully!
Total representations for retrieval: 12312
Computing embeddings...


Device set to use cpu


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

C:\Users\mihab\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\IPython\core\interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


#  6 bsp Code der angepasst ist

Schlechte Ergebnisse, aber man bekommt die Ergebnisse in der Recommendation Details als Liste.

In [ ]:
# # Cell 1: Setup and Imports
# import ast
# import threading
# from flask import Flask, request, render_template_string
# import pandas as pd
# import numpy as np
# from sentence_transformers import SentenceTransformer
# from sklearn.metrics.pairwise import cosine_similarity
# from transformers import pipeline

# app = Flask(__name__)

# # CSV file paths (all in the adjusted_datasets folder)
# CITIES_CSV = "adjusted_datasets/adjusted_cities.csv"
# FLIGHTS_CSV = "adjusted_datasets/adjusted_flights.csv"
# HOTELS_CSV = "adjusted_datasets/adjusted_hotels.csv"
# RESTAURANTS_CSV = "adjusted_datasets/adjusted_restaurants.csv"
# PREFERENCES_CSV = "adjusted_datasets/preferences.csv"
# USERS_CSV = "adjusted_datasets/users.csv"
# PASSPORTS_CSV = "adjusted_datasets/adjusted_passports.csv"
# HISTORIES_CSV = "adjusted_datasets/histories.csv"

# # Global conversation history for the chat interface
# conversation_history = []

# # ----------------------------
# # 2. Data Loading (without Neo4j)
# # ----------------------------
# def load_data():
#     # Load all CSV files into DataFrames
#     cities_df = pd.read_csv(CITIES_CSV)
#     flights_df = pd.read_csv(FLIGHTS_CSV)
#     hotels_df = pd.read_csv(HOTELS_CSV)
#     restaurants_df = pd.read_csv(RESTAURANTS_CSV)
#     preferences_df = pd.read_csv(PREFERENCES_CSV)
#     users_df = pd.read_csv(USERS_CSV)
#     passports_df = pd.read_csv(PASSPORTS_CSV)
#     histories_df = pd.read_csv(HISTORIES_CSV)
    
#     # Convert DataFrames to dictionaries for easier access
#     data = {
#         "cities": cities_df.to_dict('records'),
#         "flights": flights_df.to_dict('records'),
#         "hotels": hotels_df.to_dict('records'),
#         "restaurants": restaurants_df.to_dict('records'),
#         "preferences": preferences_df.to_dict('records'),
#         "users": users_df.to_dict('records'),
#         "passports": passports_df.to_dict('records'),
#         "histories": histories_df.to_dict('records')
#     }
#     return data

# # Load all data
# data = load_data()
# print("Data loaded successfully!")

# # ----------------------------
# # 3. Representation and Retrieval
# # ----------------------------
# def build_representation(item, fields):
#     parts = []
#     for field, label in fields.items():
#         value = item.get(field)
#         if value is not None and str(value).strip() != "":
#             parts.append(f"{label}: {value}")
#     return "; ".join(parts)

# def represent_city(city):
#     fields = {
#         "City": "City", "Country": "Country",
#         "Remote connection: Average WiFi speed (Mbps per second)": "WiFi Speed",
#         "Co-working spaces: Number of co-working spaces": "Co-working Spaces",
#         "Accommodation: Average price of 1 bedroom apartment per month": "Apartment Price",
#         "Food: Average cost of a meal at a local, mid-level restaurant": "Meal Cost",
#         "Tourist attractions: Number of Things to do on Tripadvisor": "Attractions"
#     }
#     return build_representation(city, fields)

# def represent_flight(flight):
#     fields = {
#         "Airline": "Airline", "Total Fare (EUR)": "Price",
#         "Departure Airport Code": "From", "Arrival Airport Code": "To",
#         "Duration (hrs)": "Duration", "Class": "Class"
#     }
#     return build_representation(flight, fields)

# def represent_hotel(hotel):
#     fields = {
#         "name": "Name", "price": "Price",
#         "number_reviews": "Reviews", "City": "City"
#     }
#     return build_representation(hotel, fields)

# def represent_restaurant(restaurant):
#     fields = {
#         "Restaurant Name": "Name", "Cuisines": "Cuisines",
#         "Average Cost for two": "Price for Two", "City": "City"
#     }
#     return build_representation(restaurant, fields)

# # Create representations for all data
# representations = []
# for city in data["cities"]:
#     representations.append(represent_city(city))
# for flight in data["flights"]:
#     representations.append(represent_flight(flight))
# for hotel in data["hotels"]:
#     representations.append(represent_hotel(hotel))
# for restaurant in data["restaurants"]:
#     representations.append(represent_restaurant(restaurant))

# representations = list(set(representations))
# print("Total representations for retrieval:", len(representations))

# # ----------------------------
# # 4. Embeddings and Retrieval
# # ----------------------------
# print("Computing embeddings...")
# embedder = SentenceTransformer("all-MiniLM-L6-v2")
# doc_embeddings = embedder.encode(representations, convert_to_tensor=True)

# def retrieve_documents(query, top_k=8, similarity_threshold=0.0):
#     query_embedding = embedder.encode([query], convert_to_tensor=True)
#     cos_scores = cosine_similarity(query_embedding.cpu().numpy(), doc_embeddings.cpu().numpy())[0]
#     sorted_indices = np.argsort(cos_scores)[::-1]
#     retrieved_docs = [representations[i] for i in sorted_indices[:top_k]]
#     return retrieved_docs

# # ----------------------------
# # 5. Query Processing and Generation
# # ----------------------------
# generator = pipeline(
#     "text-generation",
#     model="gpt2",
#     do_sample=True,
#     temperature=0.7,
#     max_new_tokens=200,
#     no_repeat_ngram_size=3,
#     repetition_penalty=1.2
# )

# def detect_query_type(query):
#     query_lower = query.lower()
#     if any(word in query_lower for word in ["hotel", "stay", "accommodation", "lodging"]):
#         return "hotel"
#     elif any(word in query_lower for word in ["restaurant", "eat", "dine", "food", "cuisine"]):
#         return "restaurant"
#     elif any(word in query_lower for word in ["flight", "fly", "airline", "ticket"]):
#         return "flight"
#     elif any(word in query_lower for word in ["city", "destination", "place", "visit", "location"]):
#         return "city"
#     elif any(word in query_lower for word in ["trip", "itinerary", "plan", "vacation", "holiday"]):
#         return "complete_trip"
#     elif any(word in query_lower for word in ["clear", "reset", "delete", "erase"]):
#         return "clear_history"
#     else:
#         return "general"

# def refine_query(raw_query):
#     query_type = detect_query_type(raw_query)
#     if query_type == "clear_history":
#         return raw_query, query_type
        
#     prompt = f"""
#     Refine this travel query to be more specific for a {query_type} search:
#     Original Query: {raw_query}
#     Refined Query:"""
#     result = generator(prompt, num_return_sequences=1)
#     refined = result[0]["generated_text"].replace(prompt, "").strip().split("\n")[0].strip()
#     return refined, query_type

# # ----------------------------
# # 6. Response Generation (IMPROVED)
# # ----------------------------
# def generate_response(query):
#     # First detect what kind of information the user wants
#     refined_query, query_type = refine_query(query)
#     print(f"Detected query type: {query_type}, Refined: {refined_query}")
    
#     # Handle clear history command
#     if query_type == "clear_history":
#         global conversation_history
#         conversation_history = []
#         return "I've cleared our conversation history. How can I help you with your travel plans?", []
    
#     # Retrieve relevant documents based on query type
#     retrieved_docs = retrieve_documents(refined_query, top_k=10)
    
#     if not retrieved_docs:
#         return "I couldn't find enough information about that. Could you be more specific?", []
    
#     # Generate a prompt based on query type
#     if query_type == "hotel":
#         # Find hotels matching the query (e.g., price range)
#         target_city = None
#         max_price = None
#         if "new york" in query.lower():
#             target_city = "New York"
#         if "under" in query.lower() and "$" in query.lower():
#             try:
#                 max_price = float(query.split("$")[1].split()[0])
#             except:
#                 pass
        
#         matching_hotels = []
#         for hotel in data["hotels"]:
#             if target_city and hotel.get("City", "").lower() != target_city.lower():
#                 continue
#             try:
#                 hotel_price = float(hotel.get("price", 99999))
#                 if max_price and hotel_price > max_price:
#                     continue
#                 matching_hotels.append(hotel)
#             except:
#                 continue
        
#         if not matching_hotels:
#             # Find cheapest hotel if none match the price
#             if target_city:
#                 city_hotels = [h for h in data["hotels"] if h.get("City", "").lower() == target_city.lower()]
#                 if city_hotels:
#                     try:
#                         cheapest = min(city_hotels, key=lambda x: float(x.get("price", 99999)))
#                         return f"I couldn't find hotels under ${max_price} in {target_city}. The cheapest option available is {cheapest['name']} at ${cheapest['price']}. Would you like more information about this or other options?", []
#                     except:
#                         pass
            
#             return f"I couldn't find any hotels matching your criteria in our database. Please try a different search.", []
        
#         # Sort by price
#         matching_hotels.sort(key=lambda x: float(x.get("price", 99999)))
        
#         # Build response
#         response = f"Here are the best hotel options in {target_city if target_city else 'our database'} under ${max_price if max_price else 'any price'}:\n\n"
#         for i, hotel in enumerate(matching_hotels[:5]):  # Show top 5
#             response += f"🏨 {hotel['name']}\n"
#             response += f"   - Price: ${hotel['price']}\n"
#             response += f"   - Reviews: {hotel.get('number_reviews', 'N/A')}\n"
#             response += f"   - City: {hotel.get('City', 'N/A')}\n\n"
        
#         response += "Would you like:\n"
#         response += "1. More details about any of these hotels\n"
#         response += "2. Cheaper options in a different area\n"
#         response += "3. Higher-end options with better amenities\n"
#         response += "4. Something else?"
        
#         return response, [represent_hotel(h) for h in matching_hotels[:5]]
        
#     elif query_type == "restaurant":
#         # Find restaurants matching the query
#         target_city = None
#         cuisine_type = None
        
#         # Extract city if mentioned
#         for city in data["cities"]:
#             if city['City'].lower() in query.lower():
#                 target_city = city['City']
#                 break
                
#         # Extract cuisine type if mentioned
#         cuisine_words = ["italian", "chinese", "french", "japanese", "mexican", "indian", "thai"]
#         for word in cuisine_words:
#             if word in query.lower():
#                 cuisine_type = word
#                 break
        
#         matching_restaurants = []
#         for restaurant in data["restaurants"]:
#             if target_city and restaurant.get("City", "").lower() != target_city.lower():
#                 continue
#             if cuisine_type and cuisine_type not in restaurant.get("Cuisines", "").lower():
#                 continue
#             matching_restaurants.append(restaurant)
        
#         if not matching_restaurants:
#             return f"I couldn't find any {cuisine_type + ' ' if cuisine_type else ''}restaurants matching your criteria in {target_city if target_city else 'our database'}. Please try a different search.", []
        
#         # Sort by price
#         matching_restaurants.sort(key=lambda x: float(x.get("Average Cost for two", 0)))
        
#         response = f"Here are some excellent {cuisine_type if cuisine_type else ''} restaurant options in {target_city if target_city else 'various cities'}:\n\n"
#         for i, restaurant in enumerate(matching_restaurants[:5]):
#             response += f"🍽️ {restaurant['Restaurant Name']}\n"
#             response += f"   - Cuisine: {restaurant.get('Cuisines', 'N/A')}\n"
#             response += f"   - Avg. cost for two: ${restaurant.get('Average Cost for two', 'N/A')}\n"
#             response += f"   - Location: {restaurant.get('City', 'N/A')}\n\n"
        
#         response += "Would you like to:\n"
#         response += "1. Filter by a specific price range\n"
#         response += "2. See options in a different area\n"
#         response += "3. Get recommendations for a different cuisine\n"
#         response += "4. More details about any of these"
        
#         return response, [represent_restaurant(r) for r in matching_restaurants[:5]]
        
#     elif query_type == "flight":
#         # Find flights matching the query
#         target_destination = None
#         max_price = None
        
#         # Extract destination if mentioned
#         for city in data["cities"]:
#             if city['City'].lower() in query.lower():
#                 target_destination = city['City']
#                 break
                
#         # Extract max price if mentioned
#         if "under" in query.lower() and "$" in query.lower():
#             try:
#                 max_price = float(query.split("$")[1].split()[0])
#             except:
#                 pass
        
#         matching_flights = []
#         for flight in data["flights"]:
#             if target_destination and flight.get("Arrival Airport Code", "") != target_destination:
#                 continue
#             try:
#                 flight_price = float(flight.get("Total Fare (EUR)", 99999))
#                 if max_price and flight_price > max_price:
#                     continue
#                 matching_flights.append(flight)
#             except:
#                 continue
        
#         if not matching_flights:
#             return f"I couldn't find any flights matching your criteria. Please try a different search.", []
        
#         # Sort by price
#         matching_flights.sort(key=lambda x: float(x.get("Total Fare (EUR)", 99999)))
        
#         response = f"Here are the best flight options to {target_destination if target_destination else 'various destinations'}:\n\n"
#         for i, flight in enumerate(matching_flights[:5]):
#             response += f"✈️ {flight['Airline']}\n"
#             response += f"   - From: {flight.get('Departure Airport Code', 'N/A')}\n"
#             response += f"   - To: {flight.get('Arrival Airport Code', 'N/A')}\n"
#             response += f"   - Price: ${flight.get('Total Fare (EUR)', 'N/A')}\n"
#             response += f"   - Duration: {flight.get('Duration (hrs)', 'N/A')} hours\n"
#             response += f"   - Class: {flight.get('Class', 'N/A')}\n\n"
        
#         response += "Would you like to:\n"
#         response += "1. See flights from a specific location\n"
#         response += "2. Filter by airline or flight duration\n"
#         response += "3. See business class options\n"
#         response += "4. Get recommendations for a different destination"
        
#         return response, [represent_flight(f) for f in matching_flights[:5]]
        
#     elif query_type == "city":
#         matching_cities = []
#         for city in data["cities"]:
#             matching_cities.append(city)
        
#         response = "Here are some great travel destinations:\n\n"
#         for i, city in enumerate(matching_cities[:5]):
#             response += f"🌆 {city['City']}, {city['Country']}\n"
#             response += f"   - Avg. apartment price: ${city.get('Accommodation: Average price of 1 bedroom apartment per month', 'N/A')}/month\n"
#             response += f"   - Avg. meal cost: ${city.get('Food: Average cost of a meal at a local, mid-level restaurant', 'N/A')}\n"
#             response += f"   - WiFi speed: {city.get('Remote connection: Average WiFi speed (Mbps per second)', 'N/A')} Mbps\n"
#             response += f"   - Attractions: {city.get('Tourist attractions: Number of Things to do on Tripadvisor', 'N/A')} things to do\n\n"
        
#         response += "Would you like more details about:\n"
#         response += "1. Digital nomad-friendly cities\n"
#         response += "2. Budget travel destinations\n"
#         response += "3. Luxury travel options\n"
#         response += "4. A specific city"
        
#         return response, [represent_city(c) for c in matching_cities[:5]]
        
#     elif query_type == "complete_trip":
#         # Extract destination from query
#         destination = None
#         duration = 5  # default
        
#         # Check for duration in query
#         duration_words = ["day", "week", "month"]
#         for word in duration_words:
#             if word in query.lower():
#                 try:
#                     duration = int(query.lower().split(word)[0].split()[-1])
#                     if word == "week":
#                         duration *= 7
#                     elif word == "month":
#                         duration *= 30
#                 except:
#                     pass
        
#         for city in data["cities"]:
#             if city['City'].lower() in query.lower():
#                 destination = city
#                 break
        
#         if not destination:
#             return "Please specify a destination city for your trip plan (e.g., 'Plan a 5-day trip to Paris').", []
        
#         # Get relevant items
#         city_hotels = [h for h in data["hotels"] if h.get("City", "").lower() == destination['City'].lower()]
#         city_restaurants = [r for r in data["restaurants"] if r.get("City", "").lower() == destination['City'].lower()]
#         city_flights = [f for f in data["flights"] if f.get("Arrival Airport Code", "").lower() == destination['City'].lower()]
        
#         # Build itinerary
#         attractions = destination.get('Tourist attractions: Number of Things to do on Tripadvisor', 'many')
        
#         response = f"Here's a suggested {duration}-day itinerary for {destination['City']}, {destination['Country']}:\n\n"
        
#         # Day 1: Arrival
#         response += "📅 Day 1: Arrival\n"
#         if city_flights:
#             cheapest_flight = min(city_flights, key=lambda x: float(x.get("Total Fare (EUR)", 99999)))
#             response += f"✈️ Flight: {cheapest_flight['Airline']} from {cheapest_flight['Departure Airport Code']} for ${cheapest_flight['Total Fare (EUR)']} ({cheapest_flight['Duration (hrs)']} hrs)\n"
#         if city_hotels:
#             mid_range_hotel = sorted(city_hotels, key=lambda x: float(x.get("price", 0)))[len(city_hotels)//2]
#             response += f"🏨 Hotel: {mid_range_hotel['name']} (${mid_range_hotel['price']}/night, {mid_range_hotel.get('number_reviews', 'N/A')} reviews)\n"
#         response += "   - Settle in and explore the neighborhood\n"
#         if city_restaurants:
#             local_restaurant = city_restaurants[0]
#             response += f"🍽️ Dinner: {local_restaurant['Restaurant Name']} ({local_restaurant['Cuisines']}, ${local_restaurant['Average Cost for two']} for two)\n\n"
        
#         # Day 2: Sightseeing
#         response += f"📅 Day 2: Explore {destination['City']}\n"
#         response += f"   - Visit top attractions ({attractions} options available)\n"
#         response += "   - Take a guided tour or explore on your own\n"
#         if len(city_restaurants) > 1:
#             response += f"🍽️ Dinner: {city_restaurants[1]['Restaurant Name']} ({city_restaurants[1]['Cuisines']})\n\n"
        
#         # Day 3-4: Customizable
#         response += f"📅 Day 3-{duration-1}: Customizable Activities\n"
#         response += "   - Cultural experiences\n"
#         response += "   - Day trips to nearby areas\n"
#         response += "   - Shopping and local markets\n"
#         response += "   - Relaxation time\n\n"
        
#         # Last day: Departure
#         response += f"📅 Day {duration}: Departure\n"
#         response += "   - Check out from hotel\n"
#         if city_flights:
#             response += f"✈️ Return flight: {cheapest_flight['Airline']} to {cheapest_flight['Departure Airport Code']}\n\n"
        
#         # Budget estimate
#         total_cost = 0
#         if city_flights:
#             total_cost += float(cheapest_flight['Total Fare (EUR)']) * 2  # round trip
#         if city_hotels:
#             total_cost += float(mid_range_hotel['price']) * duration
#         if city_restaurants:
#             total_cost += float(city_restaurants[0]['Average Cost for two']) * duration / 2
        
#         response += f"💰 Estimated total cost for this trip: ${total_cost:.2f} (for one person)\n\n"
        
#         response += "Would you like me to:\n"
#         response += "1. Adjust this itinerary (higher/lower budget)\n"
#         response += "2. Focus on specific interests (culture, food, adventure)\n"
#         response += "3. Provide more detailed daily activities\n"
#         response += "4. Book any of these options"
        
#         retrieved = []
#         if city_flights: retrieved.append(represent_flight(cheapest_flight))
#         if city_hotels: retrieved.append(represent_hotel(mid_range_hotel))
#         if city_restaurants: retrieved.extend([represent_restaurant(r) for r in city_restaurants[:2]])
#         retrieved.append(represent_city(destination))
        
#         return response, retrieved
        
#     else:
#         prompt = f"""Based on this travel data:
#         {retrieved_docs}
        
#         Provide a comprehensive travel answer to: {query}
#         Include relevant details about destinations, accommodations, and activities."""
        
#         result = generator(prompt, num_return_sequences=1)
#         response = result[0]["generated_text"].replace(prompt, "").strip()
        
#         return response, retrieved_docs

# # ----------------------------
# # 7. Enhanced Flask Web Interface
# # ----------------------------
# HTML_TEMPLATE = """
# <!DOCTYPE html>
# <html>
# <head>
#     <title>Enhanced Travel Assistant</title>
#     <style>
#         body {
#             font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
#             max-width: 1200px;
#             margin: 0 auto;
#             padding: 20px;
#             background-color: #f5f7fa;
#             color: #333;
#         }
#         .header {
#             background-color: #4285f4;
#             color: white;
#             padding: 20px;
#             border-radius: 8px;
#             margin-bottom: 20px;
#             text-align: center;
#         }
#         .filter-section {
#             background-color: white;
#             padding: 15px;
#             border-radius: 8px;
#             margin-bottom: 20px;
#             box-shadow: 0 2px 4px rgba(0,0,0,0.1);
#         }
#         .filter-buttons {
#             display: flex;
#             gap: 10px;
#             flex-wrap: wrap;
#             margin-top: 10px;
#         }
#         .filter-button {
#             padding: 8px 15px;
#             background-color: #e0e0e0;
#             border: none;
#             border-radius: 20px;
#             cursor: pointer;
#             transition: background-color 0.3s;
#         }
#         .filter-button:hover {
#             background-color: #d0d0d0;
#         }
#         .filter-button.active {
#             background-color: #4285f4;
#             color: white;
#         }
#         .chat-container {
#             background-color: white;
#             border-radius: 8px;
#             padding: 20px;
#             margin-bottom: 20px;
#             box-shadow: 0 2px 4px rgba(0,0,0,0.1);
#             height: 400px;
#             overflow-y: auto;
#         }
#         .message {
#             margin-bottom: 15px;
#             padding: 10px 15px;
#             border-radius: 18px;
#             max-width: 70%;
#             word-wrap: break-word;
#         }
#         .user-message {
#             background-color: #e3f2fd;
#             margin-left: auto;
#             border-bottom-right-radius: 4px;
#         }
#         .bot-message {
#             background-color: #f1f1f1;
#             margin-right: auto;
#             border-bottom-left-radius: 4px;
#         }
#         .data-section {
#             background-color: white;
#             border-radius: 8px;
#             padding: 20px;
#             margin-bottom: 20px;
#             box-shadow: 0 2px 4px rgba(0,0,0,0.1);
#         }
#         .data-item {
#             padding: 10px;
#             border-bottom: 1px solid #eee;
#         }
#         .input-section {
#             display: flex;
#             gap: 10px;
#         }
#         #user-input {
#             flex-grow: 1;
#             padding: 12px;
#             border: 1px solid #ddd;
#             border-radius: 8px;
#             font-size: 16px;
#         }
#         #submit-button {
#             padding: 12px 20px;
#             background-color: #4285f4;
#             color: white;
#             border: none;
#             border-radius: 8px;
#             cursor: pointer;
#             font-size: 16px;
#         }
#         #submit-button:hover {
#             background-color: #3367d6;
#         }
#         .query-type-indicator {
#             font-size: 14px;
#             color: #666;
#             margin-top: 5px;
#             font-style: italic;
#         }
#         .clear-button {
#             padding: 8px 15px;
#             background-color: #f44336;
#             color: white;
#             border: none;
#             border-radius: 8px;
#             cursor: pointer;
#             font-size: 14px;
#             margin-top: 10px;
#         }
#         .clear-button:hover {
#             background-color: #d32f2f;
#         }
#     </style>
# </head>
# <body>
#     <div class="header">
#         <h1>Enhanced Travel Assistant</h1>
#         <p>Get personalized travel recommendations for flights, hotels, restaurants and destinations</p>
#     </div>
    
#     <div class="filter-section">
#         <h3>Not sure what to ask? Try these:</h3>
#         <div class="filter-buttons">
#             <button class="filter-button" onclick="setQuery('Best hotels in New York under $200')">Hotels</button>
#             <button class="filter-button" onclick="setQuery('Italian restaurants in London')">Restaurants</button>
#             <button class="filter-button" onclick="setQuery('Cheapest flights to Dubai next month')">Flights</button>
#             <button class="filter-button" onclick="setQuery('Best digital nomad cities with good WiFi')">Destinations</button>
#             <button class="filter-button" onclick="setQuery('Plan a complete 5-day trip to Istanbul')">Complete Trip</button>
#         </div>
#         <button class="clear-button" onclick="setQuery('clear history')">Clear Conversation</button>
#     </div>
    
#     <div class="chat-container" id="chat-container">
#         {% for msg in history %}
#             <div class="message {% if msg.sender == 'User' %}user-message{% else %}bot-message{% endif %}">
#                 <strong>{{ msg.sender }}:</strong> {{ msg.text }}
#                 {% if msg.query_type %}
#                 <div class="query-type-indicator">Detected as: {{ msg.query_type }}</div>
#                 {% endif %}
#             </div>
#         {% endfor %}
#     </div>
    
#     <div class="data-section">
#         <h3>Recommendation Details</h3>
#         {% if retrieved_data %}
#             {% for doc in retrieved_data %}
#                 <div class="data-item">{{ doc }}</div>
#             {% endfor %}
#         {% else %}
#             <div class="data-item">No data retrieved yet. Ask about hotels, restaurants, flights or destinations.</div>
#         {% endif %}
#     </div>
    
#     <form method="post" class="input-section">
#         <input type="text" id="user-input" name="question" placeholder="Ask about hotels, flights, restaurants or destinations..." required>
#         <input type="submit" id="submit-button" value="Send">
#     </form>
    
#     <script>
#         function setQuery(query) {
#             document.getElementById('user-input').value = query;
#             if (query.toLowerCase().includes('clear')) {
#                 document.forms[0].submit();
#             }
#             document.getElementById('user-input').focus();
#         }
        
#         // Auto-scroll chat to bottom
#         var chatContainer = document.getElementById("chat-container");
#         chatContainer.scrollTop = chatContainer.scrollHeight;
#     </script>
# </body>
# </html>
# """

# @app.route("/", methods=["GET", "POST"])
# def index():
#     global conversation_history
#     retrieved_data = []
    
#     if request.method == "POST":
#         question = request.form["question"]
#         query_type = detect_query_type(question)
#         conversation_history.append({
#             "sender": "User", 
#             "text": question,
#             "query_type": query_type.replace("_", " ").title()
#         })
        
#         answer, retrieved_data = generate_response(question)
#         conversation_history.append({
#             "sender": "Assistant", 
#             "text": answer
#         })
    
#     return render_template_string(HTML_TEMPLATE, history=conversation_history, retrieved_data=retrieved_data)

# # ----------------------------
# # 8. Run the Application
# # ----------------------------
# def run_flask_app():
#     app.run(port=5001, debug=True)

# # Start the Flask app in a background thread
# threading.Thread(target=run_flask_app, daemon=True).start()
# print("Travel Assistant is running! Access it at: http://127.0.0.1:5001")

Data loaded successfully!
Total representations for retrieval: 12312
Computing embeddings...


Device set to use cpu


Travel Assistant is running! Access it at: http://127.0.0.1:5001
 * Serving Flask app '__main__'


 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
Exception in thread Thread-16 (run_flask_app):
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\mihab\AppData\Local\Temp\ipykernel_32104\2761702156.py", line 698, in run_flask_app
  File "C:\Users\mihab\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\flask\app.py", line 612, in run
    run_simple(t.cast(str, host), port, self, **options)
  File "C:\Users\mihab\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\werkzeug\serving.py"